# Search App-14 — Connect Four : le benchmark adversarial en Rust

**Navigation** : [<< App-12 ConnectFour](App-12-ConnectFour-Python.ipynb) | [Index](../README.md) | [App-34 15-puzzle](App-34-15Puzzle-Admissible-Astar-Python.ipynb) | [CSP Applications >>](App-01-NQueens-Python.ipynb) | [Version Python](App-14-ConnectFour-Adversarial-Python.ipynb) | [Version C#](App-14-ConnectFour-Adversarial-CSharp.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Comparer** les algorithmes Minimax, Alpha-Beta et MCTS sur un jeu réel
2. **mesurer** les compromis entre temps de calcul, noeuds explores et qualité de jeu
3. **Analyser** l'impact de la profondeur de recherche et du nombre d'itérations
4. **Choisir** l'algorithme adapté selon les contraintes (temps, precision)

### Prérequis
- [Search-03-Informed](../Part1-Foundations/Search-03-Informed.ipynb) : Heuristiques
- [Search-06-AdversarialSearch](../Part1-Foundations/Search-06-AdversarialSearch.ipynb) : Minimax, Alpha-Beta
- [Search-07-MCTS-And-Beyond](../Part1-Foundations/Search-07-MCTS-And-Beyond.ipynb) : MCTS (recommande)

### Durée estimee : 45 minutes

> **Source** : adapté des projets etudiants JVX (ECE 2026) et EPITA PPC 2025.
>
> **Note** : Ce notebook se concentre sur le benchmark des trois algorithmes principaux.
> Pour une étude complète — aléatoire, glouton, Minimax α-β, MCTS, framework AIMA de Russell & Norvig, approfondissement itératif et Negamax —, voir [App-12](App-12-ConnectFour-Python.ipynb).

## Hommage a des travaux etudiants

Ce notebook synthetise un projet etudiant :

**Groupe 12 -- Puissance 4** du cours EPITA SCIA *Programmation par Contraintes* 2025, realise par :
- **Leo Sambrook** ([Leo Sambrook](https://github.com/petitwhito)) -- PR #39 auteur principal (commits `1fe14b0`, `fb38116`, `d1ea607`, `3a5654a`, `c8e7aa7`, `95f4e9b`, `4d9cd75`), coordinateur du sous-projet
- **Arthur Guelennoc** ([Arthur Guelennoc](https://github.com/AntitheticalElysium)) -- Negamax, MonteCarlo, structure (commits `f2ee40a`, `e81eae0`, `993538e`, `cbd3b0b`, `4bd2c9f`, `0d92b64`, `2de673a`)
- **Arthur Hamard** ([Arthur Hamard](https://github.com/reg1um)) -- Connect4 board, Minimax/Alpha-Beta, Notebook CP/RL, DQN (commits `eae0109`, `5be7735`, `09fcb78`, `0737408`, `ab32869`, `433ef72`)

Repertoire source : [jsboigeEpita/2025-Epita-Programmation-par-Contraintes/tree/main/Groupe_12 _Puissance_4](https://github.com/jsboigeEpita/2025-Epita-Programmation-par-Contraintes/tree/main/Groupe_12%20_Puissance_4), commit principal `9c7e5ba` (merge par `jsboige`), licence **MIT** (`Copyright (c) 2025 jsboigeEpita`, fichier LICENSE dans le sous-projet, verifie firsthand 2026-09-17 via `https://raw.githubusercontent.com/.../LICENSE`).

Le projet EPITA PPC 2025 couvre déjà l'essentiel de ce que ce notebook reprend :

- l'**implementation du moteur de jeu** Puissance 4 (règles, detection de victoire, représentation du plateau) ;
- la **comparaison Minimax vs Alpha-Beta vs MCTS** sur le même terrain.

Ce qui est *ajoute* dans ce notebook :

1. la **couche benchmark rigoureuse** (Section 4) avec mesures de noeuds explores, temps, et qualité de jeu sur instances parametrees ;
2. la **discussion approfondie des compromis** (Section 5) entre profondeur et temps -- le projet source livre des implementations, pas une étude comparative formelle ;
3. la **visualisation** des resultats (Section 6), remplacee ici par ses donnees imprimees -- le carnet Rust n'ecrit aucun fichier ;
4. les **trois exercices** sur les variations realistes (profondeur iterative, iterative deepening, parallelisation).

L'enrichissement preserve les implementations du moteur et des algorithmes ; les ajouts sont concentres sur la *couche experimentale et comparative*.

**Note** : ce notebook se concentre sur le benchmark des trois algorithmes principaux. Pour une étude complete -- aléatoire, glouton, Minimax alpha-beta, MCTS, framework AIMA de Russell & Norvig, approfondissement iteratif et Negamax --, voir [App-12](App-12-ConnectFour-Python.ipynb).



## 1. Introduction

### Pourquoi le Puissance 4 ?

Le **Puissance 4** (Connect Four) est un excellent terrain de test pour les algorithmes de recherche adversariale :

| caractéristique | Valeur | Impact sur les algorithmes |
|----------------|--------|---------------------------|
| Taille du plateau | 7x6 = 42 cases | Assez grand pour etre intéressant |
| Facteur de branchement | ~4-7 | Teste l'efficacité de l'elagage |
| Profondeur moyenne | ~21 coups | suffisant pour mesurer les performances |
| Jeu résolu | Oui (1988) | Permet de vérifier l'optimalité |

### Les trois algorithmes compares

1. **Minimax** : L'algorithme de base, exploration complète de l'arbre
2. **Alpha-Beta** : Minimax avec elagage, beaucoup plus efficace
3. **MCTS** : approche probabiliste, pas besoin de fonction d'évaluation

### Questions que nous allons répondre

- Quel algorithme est le plus rapide ?
- Lequel joue le mieux avec un temps limite ?
- Comment la profondeur affecte-t-elle la qualité ?

In [1]:
// Grille du jeu et constantes du benchmark.
#[allow(dead_code)]
const ROWS: usize = 6;
#[allow(dead_code)]
const COLS: usize = 7;
#[allow(dead_code)]
const WIN_LENGTH: usize = 4;

println!("Environnement pret pour le benchmark adversarial.");


Environnement pret pour le benchmark adversarial.


## 2. Implementation du jeu

### Représentation du plateau

Nous utilisons une représentation efficace :
- Grille 7x6 stockee dans un tableau `[[i32; 7]; 6]`
- 0 = vide, 1 = joueur MAX (Rouge), -1 = joueur MIN (Jaune)
- Les coups sont indices de colonne (0 a 6)

In [2]:
// Moteur de jeu Connect Four : grille de 6 lignes x 7 colonnes.
// 0 = case vide, 1 = joueur MAX (X), -1 = joueur MIN (O).
#[allow(dead_code)]
#[derive(Clone)]
struct Board {
    grid: [[i32; 7]; 6],
    current_player: i32,
}

#[allow(dead_code)]
impl Board {
    fn new() -> Board {
        Board { grid: [[0; 7]; 6], current_player: 1 }
    }
    fn state(&self) -> Board {
        self.clone()
    }
    fn set_state(&mut self, s: Board) {
        self.grid = s.grid;
        self.current_player = s.current_player;
    }
    // Colonnes dont la case du haut est libre.
    fn valid_moves(&self) -> Vec<i32> {
        let mut moves = Vec::new();
        for c in 0..7 {
            if self.grid[0][c] == 0 {
                moves.push(c as i32);
            }
        }
        moves
    }
    // Pose le jeton par gravite ; le joueur courant change APRES la pose.
    fn drop_piece(&mut self, col: i32) -> bool {
        if col < 0 || col >= 7 {
            return false;
        }
        let c = col as usize;
        if self.grid[0][c] != 0 {
            return false;
        }
        for row in (0..6).rev() {
            if self.grid[row][c] == 0 {
                self.grid[row][c] = self.current_player;
                self.current_player *= -1;
                return true;
            }
        }
        false
    }
    // Retire le jeton du haut de la colonne ; le joueur courant change aussi.
    fn undo_move(&mut self, col: i32) {
        let c = col as usize;
        for row in 0..6 {
            if self.grid[row][c] != 0 {
                self.grid[row][c] = 0;
                self.current_player *= -1;
                return;
            }
        }
    }
    // Ordre des directions : horizontales, verticales, diagonales bas-droite,
    // diagonales bas-gauche.
    fn check_winner(&self) -> i32 {
        for r in 0..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r][c + 1], self.grid[r][c + 2], self.grid[r][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..7 {
                let w = [self.grid[r][c], self.grid[r + 1][c], self.grid[r + 2][c], self.grid[r + 3][c]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r + 1][c + 1], self.grid[r + 2][c + 2], self.grid[r + 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 3..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r - 1][c + 1], self.grid[r - 2][c + 2], self.grid[r - 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        0
    }
    fn is_terminal(&self) -> bool {
        self.check_winner() != 0 || self.valid_moves().is_empty()
    }
    fn get_utility(&self, maximizing_player: i32) -> f64 {
        (self.check_winner() * maximizing_player) as f64
    }
    fn display(&self) -> String {
        let sym = |c: i32| -> char {
            if c == 0 {
                '.'
            } else if c == 1 {
                'X'
            } else {
                'O'
            }
        };
        let mut lines: Vec<String> = Vec::new();
        for r in 0..6 {
            let cells: Vec<String> = (0..7).map(|c| sym(self.grid[r][c]).to_string()).collect();
            lines.push(cells.join(" "));
        }
        let cols: Vec<String> = (0..7).map(|c| c.to_string()).collect();
        lines.push(cols.join(" "));
        lines.join("\n")
    }
}
// Separateur de milliers, comme le format `:,` de Python.
#[allow(dead_code)]
fn thousands(n: u64) -> String {
    let s = n.to_string();
    let len = s.len();
    let mut out = String::new();
    for (i, ch) in s.chars().enumerate() {
        if i > 0 && (len - i) % 3 == 0 {
            out.push(',');
        }
        out.push(ch);
    }
    out
}

// Liste d'entiers au format de la repr() d'une liste Python.
#[allow(dead_code)]
fn py_list_i32(values: &[i32]) -> String {
    let items: Vec<String> = values.iter().map(|v| v.to_string()).collect();
    format!("[{}]", items.join(", "))
}

// Un Option imprime comme un Optional Python.
#[allow(dead_code)]
fn py_opt_i32(value: Option<i32>) -> String {
    match value {
        Some(v) => v.to_string(),
        None => "None".to_string(),
    }
}
let game = Board::new();
println!("Plateau initial:");
println!("{}", game.display());
println!();
println!(
    "Joueur actuel: {}",
    if game.current_player == 1 { "MAX (X)" } else { "MIN (O)" }
);
println!("Coups valides: {}", py_list_i32(&game.valid_moves()));


Plateau initial:


. . . . . . .


. . . . . . .


. . . . . . .


. . . . . . .


. . . . . . .


. . . . . . .


0 1 2 3 4 5 6


Joueur actuel: MAX (X)


Coups valides: [0, 1, 2, 3, 4, 5, 6]


Test du moteur de jeu avec une partie courte.

In [3]:
// Moteur de jeu Connect Four : grille de 6 lignes x 7 colonnes.
// 0 = case vide, 1 = joueur MAX (X), -1 = joueur MIN (O).
#[allow(dead_code)]
#[derive(Clone)]
struct Board {
    grid: [[i32; 7]; 6],
    current_player: i32,
}

#[allow(dead_code)]
impl Board {
    fn new() -> Board {
        Board { grid: [[0; 7]; 6], current_player: 1 }
    }
    fn state(&self) -> Board {
        self.clone()
    }
    fn set_state(&mut self, s: Board) {
        self.grid = s.grid;
        self.current_player = s.current_player;
    }
    // Colonnes dont la case du haut est libre.
    fn valid_moves(&self) -> Vec<i32> {
        let mut moves = Vec::new();
        for c in 0..7 {
            if self.grid[0][c] == 0 {
                moves.push(c as i32);
            }
        }
        moves
    }
    // Pose le jeton par gravite ; le joueur courant change APRES la pose.
    fn drop_piece(&mut self, col: i32) -> bool {
        if col < 0 || col >= 7 {
            return false;
        }
        let c = col as usize;
        if self.grid[0][c] != 0 {
            return false;
        }
        for row in (0..6).rev() {
            if self.grid[row][c] == 0 {
                self.grid[row][c] = self.current_player;
                self.current_player *= -1;
                return true;
            }
        }
        false
    }
    // Retire le jeton du haut de la colonne ; le joueur courant change aussi.
    fn undo_move(&mut self, col: i32) {
        let c = col as usize;
        for row in 0..6 {
            if self.grid[row][c] != 0 {
                self.grid[row][c] = 0;
                self.current_player *= -1;
                return;
            }
        }
    }
    // Ordre des directions : horizontales, verticales, diagonales bas-droite,
    // diagonales bas-gauche.
    fn check_winner(&self) -> i32 {
        for r in 0..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r][c + 1], self.grid[r][c + 2], self.grid[r][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..7 {
                let w = [self.grid[r][c], self.grid[r + 1][c], self.grid[r + 2][c], self.grid[r + 3][c]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r + 1][c + 1], self.grid[r + 2][c + 2], self.grid[r + 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 3..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r - 1][c + 1], self.grid[r - 2][c + 2], self.grid[r - 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        0
    }
    fn is_terminal(&self) -> bool {
        self.check_winner() != 0 || self.valid_moves().is_empty()
    }
    fn get_utility(&self, maximizing_player: i32) -> f64 {
        (self.check_winner() * maximizing_player) as f64
    }
    fn display(&self) -> String {
        let sym = |c: i32| -> char {
            if c == 0 {
                '.'
            } else if c == 1 {
                'X'
            } else {
                'O'
            }
        };
        let mut lines: Vec<String> = Vec::new();
        for r in 0..6 {
            let cells: Vec<String> = (0..7).map(|c| sym(self.grid[r][c]).to_string()).collect();
            lines.push(cells.join(" "));
        }
        let cols: Vec<String> = (0..7).map(|c| c.to_string()).collect();
        lines.push(cols.join(" "));
        lines.join("\n")
    }
}
let mut game = Board::new();
for &mv in [3i32, 3, 4, 2, 2, 4, 1, 5, 5].iter() {
    game.drop_piece(mv);
}

println!("Plateau apres quelques coups:");
println!("{}", game.display());
println!();
println!("Gagnant: {}", game.check_winner());


Plateau apres quelques coups:


. . . . . . .


. . . . . . .


. . . . . . .


. . . . . . .


. . X O O X .


. X O X X O .


0 1 2 3 4 5 6


Gagnant: 0


### Interprétation : Test du moteur de jeu

**Sortie obtenue** : Partie test après 9 coups [3, 3, 4, 2, 2, 4, 1, 5, 5].

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| Coups joues | 9 | Partie en cours (42 coups max) |
| Jetons MAX (X) | 5 | X a joue en 1er, 3eme, 5eme, 7eme, 9eme |
| Jetons MIN (O) | 4 | O a joue en 2eme, 4eme, 6eme, 8eme |
| Gagnant | 0 | Pas de gagnant (partie non terminee) |
| Coups valides restants | 7 | Toutes les colonnes sont encore jouables |

**Position finale** :
```
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . X O O X .
. X O X X O .
0 1 2 3 4 5 6
```

**Points clés** :
1. **Gravite correcte** : Les jetons tombent jusqu'au premier emplacement vide
2. **Alternance respectee** : MAX et MIN alternent correctement
3. **détection victoire** : Aucun alignement de 4 (correct)
4. **Moteur fonctionnel** : Toutes les méthodes de base opérationnelles

> **Note technique** : La sequence de coups [3, 3, 4, 2, 2, 4, 1, 5, 5] créé une position intéressante avec plusieurs menaces potentielles. MIN a 3 jetons alignes horizontalement (colonnes 3-4-5) mais sans la 4eme case. MAX a aussi des opportunities. Cette position sert de base pour tester les algorithmes de recherche qui vont évaluer quel coup est le meilleur pour chaque joueur.

## 3. Algorithme Minimax

**Rappel du principe**

Minimax explore l'arbre de jeu complet jusqu'a une profondeur donnée :
- **MAX** cherche a maximiser l'utilité
- **MIN** cherche a minimiser l'utilité

### Fonction d'évaluation

Pour les états non terminaux, nous utilisons une heuristique basee sur :
- Le contrôle du centre
- Le nombre de "fenêtrès" gagnantes potentielles

In [4]:
// Moteur de jeu Connect Four : grille de 6 lignes x 7 colonnes.
// 0 = case vide, 1 = joueur MAX (X), -1 = joueur MIN (O).
#[allow(dead_code)]
#[derive(Clone)]
struct Board {
    grid: [[i32; 7]; 6],
    current_player: i32,
}

#[allow(dead_code)]
impl Board {
    fn new() -> Board {
        Board { grid: [[0; 7]; 6], current_player: 1 }
    }
    fn state(&self) -> Board {
        self.clone()
    }
    fn set_state(&mut self, s: Board) {
        self.grid = s.grid;
        self.current_player = s.current_player;
    }
    // Colonnes dont la case du haut est libre.
    fn valid_moves(&self) -> Vec<i32> {
        let mut moves = Vec::new();
        for c in 0..7 {
            if self.grid[0][c] == 0 {
                moves.push(c as i32);
            }
        }
        moves
    }
    // Pose le jeton par gravite ; le joueur courant change APRES la pose.
    fn drop_piece(&mut self, col: i32) -> bool {
        if col < 0 || col >= 7 {
            return false;
        }
        let c = col as usize;
        if self.grid[0][c] != 0 {
            return false;
        }
        for row in (0..6).rev() {
            if self.grid[row][c] == 0 {
                self.grid[row][c] = self.current_player;
                self.current_player *= -1;
                return true;
            }
        }
        false
    }
    // Retire le jeton du haut de la colonne ; le joueur courant change aussi.
    fn undo_move(&mut self, col: i32) {
        let c = col as usize;
        for row in 0..6 {
            if self.grid[row][c] != 0 {
                self.grid[row][c] = 0;
                self.current_player *= -1;
                return;
            }
        }
    }
    // Ordre des directions : horizontales, verticales, diagonales bas-droite,
    // diagonales bas-gauche.
    fn check_winner(&self) -> i32 {
        for r in 0..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r][c + 1], self.grid[r][c + 2], self.grid[r][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..7 {
                let w = [self.grid[r][c], self.grid[r + 1][c], self.grid[r + 2][c], self.grid[r + 3][c]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r + 1][c + 1], self.grid[r + 2][c + 2], self.grid[r + 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 3..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r - 1][c + 1], self.grid[r - 2][c + 2], self.grid[r - 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        0
    }
    fn is_terminal(&self) -> bool {
        self.check_winner() != 0 || self.valid_moves().is_empty()
    }
    fn get_utility(&self, maximizing_player: i32) -> f64 {
        (self.check_winner() * maximizing_player) as f64
    }
    fn display(&self) -> String {
        let sym = |c: i32| -> char {
            if c == 0 {
                '.'
            } else if c == 1 {
                'X'
            } else {
                'O'
            }
        };
        let mut lines: Vec<String> = Vec::new();
        for r in 0..6 {
            let cells: Vec<String> = (0..7).map(|c| sym(self.grid[r][c]).to_string()).collect();
            lines.push(cells.join(" "));
        }
        let cols: Vec<String> = (0..7).map(|c| c.to_string()).collect();
        lines.push(cols.join(" "));
        lines.join("\n")
    }
}
// Heuristique : score des fenetres de 4 cases + controle du centre.
#[allow(dead_code)]
fn evaluate_window(window: &[i32; 4], player: i32) -> i64 {
    let mut score: i64 = 0;
    let opponent = -player;

    let player_count = window.iter().filter(|&&x| x == player).count();
    let opponent_count = window.iter().filter(|&&x| x == opponent).count();
    let empty_count = window.iter().filter(|&&x| x == 0).count();

    // Fenetre gagnante
    if player_count == 4 {
        score += 100;
    } else if player_count == 3 && empty_count == 1 {
        score += 5;
    } else if player_count == 2 && empty_count == 2 {
        score += 2;
    }

    // Bloquer l'adversaire (3 jetons adverses + 1 vide)
    if opponent_count == 3 && empty_count == 1 {
        score -= 4;
    }

    score
}

#[allow(dead_code)]
fn evaluate_position(game: &Board, player: i32) -> f64 {
    if game.is_terminal() {
        return game.get_utility(player) * 1000.0;
    }

    let mut score: i64 = 0;

    // Bonus pour le centre (colonne 3)
    for r in 0..6 {
        if game.grid[r][3] == player {
            score += 3;
        }
    }

    // Horizontales
    for r in 0..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r][c + 1], game.grid[r][c + 2], game.grid[r][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Verticales
    for r in 0..3 {
        for c in 0..7 {
            let w = [game.grid[r][c], game.grid[r + 1][c], game.grid[r + 2][c], game.grid[r + 3][c]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-droite
    for r in 0..3 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r + 1][c + 1], game.grid[r + 2][c + 2], game.grid[r + 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-gauche
    for r in 3..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r - 1][c + 1], game.grid[r - 2][c + 2], game.grid[r - 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    score as f64
}
let mut game = Board::new();
for &mv in [3i32, 3, 4].iter() {
    game.drop_piece(mv);
}

println!("{}", game.display());
println!();
println!("Score pour MAX: {}", evaluate_position(&game, 1));
println!("Score pour MIN: {}", evaluate_position(&game, -1));


. . . . . . .


. . . . . . .


. . . . . . .


. . . . . . .


. . . O . . .


. . . X X . .


0 1 2 3 4 5 6


Score pour MAX: 9


Score pour MIN: 3


### Interprétation : Test de la fonction d'évaluation

**Sortie obtenue** : évaluation d'une position avec 3 jetons (MAX joue au centre).

| Joueur | Score | Analyse |
|---------|-------|---------|
| MAX (X) | 9 | Contrôle du centre + 2 jetons alignes horizontalement |
| MIN (O) | 3 | 1 seul jeton, pas de menace immédiate |

**Position obtenue** après les coups [3, 3, 4] :
```
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . O . . .
. . . X X . .
0 1 2 3 4 5 6
```

**Points clés** :
1. **Avantage MAX** : Score 9 vs 3 = MAX a un net avantage
2. **Contrôle du centre** : Les 2 jetons X au centre (colonnes 3-4) sont valorises
3. **Bonus centre** : La colonne 3 a 1 jeton X = bonus de 3 points
4. **Potentiel d'alignement** : 2 X horizontaux (cases 3-4) créent une menace

> **Note technique** : L'heuristique combine plusieurs facteurs : (1) bonus centre = +3 par jeton en colonne 3, (2) fenêtrès gagnantes (4 alignes) = +100, (3) 3 jetons + 1 vide = +5, (4) 2 jetons + 2 vides = +2, (5) bloquer adversaire = -4. Ici, MAX a 2 fenêtrès avec 2 jetons (colonnes 2-3-4-5 et 3-4-5-6) = 2x2 = 4 points, plus 3 points de centre = 7, plus d'autrès contributions = 9 total.

Statistiques de benchmark pour comparer les algorithmes.

In [5]:
// Moteur de jeu Connect Four : grille de 6 lignes x 7 colonnes.
// 0 = case vide, 1 = joueur MAX (X), -1 = joueur MIN (O).
#[allow(dead_code)]
#[derive(Clone)]
struct Board {
    grid: [[i32; 7]; 6],
    current_player: i32,
}

#[allow(dead_code)]
impl Board {
    fn new() -> Board {
        Board { grid: [[0; 7]; 6], current_player: 1 }
    }
    fn state(&self) -> Board {
        self.clone()
    }
    fn set_state(&mut self, s: Board) {
        self.grid = s.grid;
        self.current_player = s.current_player;
    }
    // Colonnes dont la case du haut est libre.
    fn valid_moves(&self) -> Vec<i32> {
        let mut moves = Vec::new();
        for c in 0..7 {
            if self.grid[0][c] == 0 {
                moves.push(c as i32);
            }
        }
        moves
    }
    // Pose le jeton par gravite ; le joueur courant change APRES la pose.
    fn drop_piece(&mut self, col: i32) -> bool {
        if col < 0 || col >= 7 {
            return false;
        }
        let c = col as usize;
        if self.grid[0][c] != 0 {
            return false;
        }
        for row in (0..6).rev() {
            if self.grid[row][c] == 0 {
                self.grid[row][c] = self.current_player;
                self.current_player *= -1;
                return true;
            }
        }
        false
    }
    // Retire le jeton du haut de la colonne ; le joueur courant change aussi.
    fn undo_move(&mut self, col: i32) {
        let c = col as usize;
        for row in 0..6 {
            if self.grid[row][c] != 0 {
                self.grid[row][c] = 0;
                self.current_player *= -1;
                return;
            }
        }
    }
    // Ordre des directions : horizontales, verticales, diagonales bas-droite,
    // diagonales bas-gauche.
    fn check_winner(&self) -> i32 {
        for r in 0..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r][c + 1], self.grid[r][c + 2], self.grid[r][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..7 {
                let w = [self.grid[r][c], self.grid[r + 1][c], self.grid[r + 2][c], self.grid[r + 3][c]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r + 1][c + 1], self.grid[r + 2][c + 2], self.grid[r + 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 3..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r - 1][c + 1], self.grid[r - 2][c + 2], self.grid[r - 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        0
    }
    fn is_terminal(&self) -> bool {
        self.check_winner() != 0 || self.valid_moves().is_empty()
    }
    fn get_utility(&self, maximizing_player: i32) -> f64 {
        (self.check_winner() * maximizing_player) as f64
    }
    fn display(&self) -> String {
        let sym = |c: i32| -> char {
            if c == 0 {
                '.'
            } else if c == 1 {
                'X'
            } else {
                'O'
            }
        };
        let mut lines: Vec<String> = Vec::new();
        for r in 0..6 {
            let cells: Vec<String> = (0..7).map(|c| sym(self.grid[r][c]).to_string()).collect();
            lines.push(cells.join(" "));
        }
        let cols: Vec<String> = (0..7).map(|c| c.to_string()).collect();
        lines.push(cols.join(" "));
        lines.join("\n")
    }
}
// Heuristique : score des fenetres de 4 cases + controle du centre.
#[allow(dead_code)]
fn evaluate_window(window: &[i32; 4], player: i32) -> i64 {
    let mut score: i64 = 0;
    let opponent = -player;

    let player_count = window.iter().filter(|&&x| x == player).count();
    let opponent_count = window.iter().filter(|&&x| x == opponent).count();
    let empty_count = window.iter().filter(|&&x| x == 0).count();

    // Fenetre gagnante
    if player_count == 4 {
        score += 100;
    } else if player_count == 3 && empty_count == 1 {
        score += 5;
    } else if player_count == 2 && empty_count == 2 {
        score += 2;
    }

    // Bloquer l'adversaire (3 jetons adverses + 1 vide)
    if opponent_count == 3 && empty_count == 1 {
        score -= 4;
    }

    score
}

#[allow(dead_code)]
fn evaluate_position(game: &Board, player: i32) -> f64 {
    if game.is_terminal() {
        return game.get_utility(player) * 1000.0;
    }

    let mut score: i64 = 0;

    // Bonus pour le centre (colonne 3)
    for r in 0..6 {
        if game.grid[r][3] == player {
            score += 3;
        }
    }

    // Horizontales
    for r in 0..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r][c + 1], game.grid[r][c + 2], game.grid[r][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Verticales
    for r in 0..3 {
        for c in 0..7 {
            let w = [game.grid[r][c], game.grid[r + 1][c], game.grid[r + 2][c], game.grid[r + 3][c]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-droite
    for r in 0..3 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r + 1][c + 1], game.grid[r + 2][c + 2], game.grid[r + 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-gauche
    for r in 3..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r - 1][c + 1], game.grid[r - 2][c + 2], game.grid[r - 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    score as f64
}
// Statistiques de recherche et Minimax (sans tri des coups).
#[allow(dead_code)]
#[derive(Default)]
struct Stats {
    nodes: u64,
}

// Minimax : les coups sont parcourus dans l'ordre brut des colonnes,
// et le premier maximum strict est retenu.
#[allow(dead_code)]
fn minimax(
    game: &mut Board,
    depth: i32,
    maximizing: bool,
    stats: &mut Stats,
    player: i32,
) -> (f64, Option<i32>) {
    stats.nodes += 1;

    if depth == 0 || game.is_terminal() {
        if game.is_terminal() {
            return (game.get_utility(player) * 1000.0, None);
        }
        return (evaluate_position(game, player), None);
    }

    let valid_moves = game.valid_moves();

    if maximizing {
        let mut best_value = f64::NEG_INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, false, stats, player);
            game.undo_move(mv);
            if value > best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    } else {
        let mut best_value = f64::INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, true, stats, player);
            game.undo_move(mv);
            if value < best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    }
}
// Separateur de milliers, comme le format `:,` de Python.
#[allow(dead_code)]
fn thousands(n: u64) -> String {
    let s = n.to_string();
    let len = s.len();
    let mut out = String::new();
    for (i, ch) in s.chars().enumerate() {
        if i > 0 && (len - i) % 3 == 0 {
            out.push(',');
        }
        out.push(ch);
    }
    out
}

// Liste d'entiers au format de la repr() d'une liste Python.
#[allow(dead_code)]
fn py_list_i32(values: &[i32]) -> String {
    let items: Vec<String> = values.iter().map(|v| v.to_string()).collect();
    format!("[{}]", items.join(", "))
}

// Un Option imprime comme un Optional Python.
#[allow(dead_code)]
fn py_opt_i32(value: Option<i32>) -> String {
    match value {
        Some(v) => v.to_string(),
        None => "None".to_string(),
    }
}
let mut game = Board::new();
let mut stats = Stats { nodes: 0 };

let start = std::time::Instant::now();
let (value, mv) = minimax(&mut game, 4, true, &mut stats, 1);
let elapsed = start.elapsed().as_secs_f64();

println!("Minimax (profondeur 4):");
println!("  Meilleur coup: {}", py_opt_i32(mv));
println!("  Valeur: {:.2}", value);
println!("  Noeuds explores: {}", thousands(stats.nodes));
println!("  Temps: {:.4}s", elapsed);


Minimax (profondeur 4):


  Meilleur coup: 3


  Valeur: 6.00


  Noeuds explores: 2,801


  Temps: 0.0022s


### Interprétation : Test Minimax

**Sortie obtenue** : Minimax avec profondeur 4 sur position initiale.

| Métrique | Valeur | Signification |
|----------|--------|---------------|
| Meilleur coup | 3 (colonne centrale) | Contrôle du centre = stratégie optimale |
| Valeur | 6.00 | Score heuristique positif (avantage MAX) |
| Noeuds explores | 2,801 | Explosion combinatoire : somme des niveaux 0 a 4 (1 + 7 + 49 + 343 + 2 401) |
| Temps | Runtime *machine-dep* | Performance typique pour d=4 (Rust, noyau evcxr) |

**Points clés** :
1. **Coup optimal** : Colonne 3 (centre) est le meilleur premier coup
2. **Complexité** : 2,801 noeuds pour seulement 4 profondeurs
3. **Heuristique efficace** : Score 6.00 = avantage modere pour MAX
4. **Scalabilite** : les benchmarks ulterieurs (cellule suivante) montrent que d>5 devient prohibitif (croissance b^d)

> **Note technique** : Le **nombre de noeuds** (2,801) est un invariant structurel du problème sur cette instance (déterministe sur la profondeur et la position initiale) : il est **superieur a 7^4 = 2,401** parce qu'il compte **tous les noeuds des niveaux 0 a 4** (1 + 7 + 49 + 343 + 2 401), alors que 7^4 = 2,401 ne compte que les **feuilles** du dernier niveau : le facteur de branchement ne diminue pas, Minimax elague a la profondeur demandee (d=4 ici) et n'explore aucune branche plus longue. Minimax explore systematiquement TOUTES les branches jusqu'a la profondeur d, sans aucun elagage. C'est son principal defaut. Le **temps d'exécution** est en revanche machine-dep (CPython + charge système + taille instance) ; il **ne survit pas** a une re-exécution sur une autre machine. Pour observer vos propres timings, executez la cellule de benchmark ci-dessous.



## 4. Algorithme Alpha-Beta

### Principe de l'elagage

Alpha-Beta améliore Minimax en eliminant les branches inutiles :
- **alpha** : Meilleure valeur que MAX peut garantir
- **beta** : Meilleure valeur que MIN peut garantir

Si alpha >= beta, on peut couper la branche (l'adversaire ne permettra jamais d'atteindre cette branche).

### Gain théorique

Avec un ordonnancement optimal des coups : **O(b^(d/2))** au lieu de **O(b^d)**.

In [6]:
// Moteur de jeu Connect Four : grille de 6 lignes x 7 colonnes.
// 0 = case vide, 1 = joueur MAX (X), -1 = joueur MIN (O).
#[allow(dead_code)]
#[derive(Clone)]
struct Board {
    grid: [[i32; 7]; 6],
    current_player: i32,
}

#[allow(dead_code)]
impl Board {
    fn new() -> Board {
        Board { grid: [[0; 7]; 6], current_player: 1 }
    }
    fn state(&self) -> Board {
        self.clone()
    }
    fn set_state(&mut self, s: Board) {
        self.grid = s.grid;
        self.current_player = s.current_player;
    }
    // Colonnes dont la case du haut est libre.
    fn valid_moves(&self) -> Vec<i32> {
        let mut moves = Vec::new();
        for c in 0..7 {
            if self.grid[0][c] == 0 {
                moves.push(c as i32);
            }
        }
        moves
    }
    // Pose le jeton par gravite ; le joueur courant change APRES la pose.
    fn drop_piece(&mut self, col: i32) -> bool {
        if col < 0 || col >= 7 {
            return false;
        }
        let c = col as usize;
        if self.grid[0][c] != 0 {
            return false;
        }
        for row in (0..6).rev() {
            if self.grid[row][c] == 0 {
                self.grid[row][c] = self.current_player;
                self.current_player *= -1;
                return true;
            }
        }
        false
    }
    // Retire le jeton du haut de la colonne ; le joueur courant change aussi.
    fn undo_move(&mut self, col: i32) {
        let c = col as usize;
        for row in 0..6 {
            if self.grid[row][c] != 0 {
                self.grid[row][c] = 0;
                self.current_player *= -1;
                return;
            }
        }
    }
    // Ordre des directions : horizontales, verticales, diagonales bas-droite,
    // diagonales bas-gauche.
    fn check_winner(&self) -> i32 {
        for r in 0..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r][c + 1], self.grid[r][c + 2], self.grid[r][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..7 {
                let w = [self.grid[r][c], self.grid[r + 1][c], self.grid[r + 2][c], self.grid[r + 3][c]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r + 1][c + 1], self.grid[r + 2][c + 2], self.grid[r + 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 3..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r - 1][c + 1], self.grid[r - 2][c + 2], self.grid[r - 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        0
    }
    fn is_terminal(&self) -> bool {
        self.check_winner() != 0 || self.valid_moves().is_empty()
    }
    fn get_utility(&self, maximizing_player: i32) -> f64 {
        (self.check_winner() * maximizing_player) as f64
    }
    fn display(&self) -> String {
        let sym = |c: i32| -> char {
            if c == 0 {
                '.'
            } else if c == 1 {
                'X'
            } else {
                'O'
            }
        };
        let mut lines: Vec<String> = Vec::new();
        for r in 0..6 {
            let cells: Vec<String> = (0..7).map(|c| sym(self.grid[r][c]).to_string()).collect();
            lines.push(cells.join(" "));
        }
        let cols: Vec<String> = (0..7).map(|c| c.to_string()).collect();
        lines.push(cols.join(" "));
        lines.join("\n")
    }
}
// Heuristique : score des fenetres de 4 cases + controle du centre.
#[allow(dead_code)]
fn evaluate_window(window: &[i32; 4], player: i32) -> i64 {
    let mut score: i64 = 0;
    let opponent = -player;

    let player_count = window.iter().filter(|&&x| x == player).count();
    let opponent_count = window.iter().filter(|&&x| x == opponent).count();
    let empty_count = window.iter().filter(|&&x| x == 0).count();

    // Fenetre gagnante
    if player_count == 4 {
        score += 100;
    } else if player_count == 3 && empty_count == 1 {
        score += 5;
    } else if player_count == 2 && empty_count == 2 {
        score += 2;
    }

    // Bloquer l'adversaire (3 jetons adverses + 1 vide)
    if opponent_count == 3 && empty_count == 1 {
        score -= 4;
    }

    score
}

#[allow(dead_code)]
fn evaluate_position(game: &Board, player: i32) -> f64 {
    if game.is_terminal() {
        return game.get_utility(player) * 1000.0;
    }

    let mut score: i64 = 0;

    // Bonus pour le centre (colonne 3)
    for r in 0..6 {
        if game.grid[r][3] == player {
            score += 3;
        }
    }

    // Horizontales
    for r in 0..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r][c + 1], game.grid[r][c + 2], game.grid[r][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Verticales
    for r in 0..3 {
        for c in 0..7 {
            let w = [game.grid[r][c], game.grid[r + 1][c], game.grid[r + 2][c], game.grid[r + 3][c]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-droite
    for r in 0..3 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r + 1][c + 1], game.grid[r + 2][c + 2], game.grid[r + 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-gauche
    for r in 3..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r - 1][c + 1], game.grid[r - 2][c + 2], game.grid[r - 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    score as f64
}
// Statistiques de recherche et Minimax (sans tri des coups).
#[allow(dead_code)]
#[derive(Default)]
struct Stats {
    nodes: u64,
}

// Minimax : les coups sont parcourus dans l'ordre brut des colonnes,
// et le premier maximum strict est retenu.
#[allow(dead_code)]
fn minimax(
    game: &mut Board,
    depth: i32,
    maximizing: bool,
    stats: &mut Stats,
    player: i32,
) -> (f64, Option<i32>) {
    stats.nodes += 1;

    if depth == 0 || game.is_terminal() {
        if game.is_terminal() {
            return (game.get_utility(player) * 1000.0, None);
        }
        return (evaluate_position(game, player), None);
    }

    let valid_moves = game.valid_moves();

    if maximizing {
        let mut best_value = f64::NEG_INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, false, stats, player);
            game.undo_move(mv);
            if value > best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    } else {
        let mut best_value = f64::INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, true, stats, player);
            game.undo_move(mv);
            if value < best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    }
}
// Alpha-Beta : les coups sont ordonnes par proximite au centre (tri stable),
// et la coupe s'applique quand beta <= alpha.
#[allow(dead_code)]
fn alpha_beta(
    game: &mut Board,
    depth: i32,
    mut alpha: f64,
    mut beta: f64,
    maximizing: bool,
    stats: &mut Stats,
    player: i32,
) -> (f64, Option<i32>) {
    stats.nodes += 1;

    if depth == 0 || game.is_terminal() {
        if game.is_terminal() {
            return (game.get_utility(player) * 1000.0, None);
        }
        return (evaluate_position(game, player), None);
    }

    let mut valid_moves = game.valid_moves();
    valid_moves.sort_by_key(|&c| (c - 3).abs());

    if maximizing {
        let mut best_value = f64::NEG_INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = alpha_beta(game, depth - 1, alpha, beta, false, stats, player);
            game.undo_move(mv);
            if value > best_value {
                best_value = value;
                best_move = Some(mv);
            }
            if best_value > alpha {
                alpha = best_value;
            }
            if beta <= alpha {
                break;
            }
        }
        (best_value, best_move)
    } else {
        let mut best_value = f64::INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = alpha_beta(game, depth - 1, alpha, beta, true, stats, player);
            game.undo_move(mv);
            if value < best_value {
                best_value = value;
                best_move = Some(mv);
            }
            if best_value < beta {
                beta = best_value;
            }
            if beta <= alpha {
                break;
            }
        }
        (best_value, best_move)
    }
}
// Separateur de milliers, comme le format `:,` de Python.
#[allow(dead_code)]
fn thousands(n: u64) -> String {
    let s = n.to_string();
    let len = s.len();
    let mut out = String::new();
    for (i, ch) in s.chars().enumerate() {
        if i > 0 && (len - i) % 3 == 0 {
            out.push(',');
        }
        out.push(ch);
    }
    out
}

// Liste d'entiers au format de la repr() d'une liste Python.
#[allow(dead_code)]
fn py_list_i32(values: &[i32]) -> String {
    let items: Vec<String> = values.iter().map(|v| v.to_string()).collect();
    format!("[{}]", items.join(", "))
}

// Un Option imprime comme un Optional Python.
#[allow(dead_code)]
fn py_opt_i32(value: Option<i32>) -> String {
    match value {
        Some(v) => v.to_string(),
        None => "None".to_string(),
    }
}
let mut game = Board::new();
let mut stats_ab = Stats { nodes: 0 };

let start = std::time::Instant::now();
let (value_ab, mv_ab) = alpha_beta(
    &mut game,
    4,
    f64::NEG_INFINITY,
    f64::INFINITY,
    true,
    &mut stats_ab,
    1,
);
let elapsed = start.elapsed().as_secs_f64();

println!("Alpha-Beta (profondeur 4):");
println!("  Meilleur coup: {}", py_opt_i32(mv_ab));
println!("  Valeur: {:.2}", value_ab);
println!("  Noeuds explores: {}", thousands(stats_ab.nodes));
println!("  Temps: {:.4}s", elapsed);

// Chaque cellule est autonome : le minimax est rejoue ici pour obtenir le
// compte de noeuds qui donne le ratio de reduction.
let mut game_mm = Board::new();
let mut stats_mm = Stats { nodes: 0 };
let _ = minimax(&mut game_mm, 4, true, &mut stats_mm, 1);

println!();
println!(
    "Reduction vs Minimax: {:.1}x",
    stats_mm.nodes as f64 / stats_ab.nodes as f64
);


Alpha-Beta (profondeur 4):


  Meilleur coup: 3


  Valeur: 6.00


  Noeuds explores: 178


  Temps: 0.0001s


Reduction vs Minimax: 15.7x


### Interprétation : Test Alpha-Beta

**Sortie obtenue** : Alpha-Beta avec profondeur 4 sur position initiale.

| Métrique | Valeur | Comparaison vs Minimax |
|----------|--------|----------------------|
| Meilleur coup | 3 | Identique (centre) |
| Valeur | 6.00 | Identique (optimal) |
| Noeuds explores | 178 | **15.7x moins** (invariant structurel) |
| Temps | Runtime *machine-dep* | Speedup wall-clock *machine-dep* (Minimax / Alpha-Beta) |

**Points clés** :
1. **résultat identique** : même coup (colonne 3) et même valeur (6.00)
2. **Elagage massif** : 178 noeuds vs 2,801 = 94% de reduction (invariant structurel)
3. **Optimalité gardee** : Alpha-Beta ne change pas le résultat, seulement le coût de la recherche

> **Note technique** : La **reduction de 15.7x en noeuds** a profondeur 4 est un invariant structurel de l'elagage Alpha-Beta sur cette instance (déterministe, propriete algorithmique). Elle est excellente mais loin du maximum théorique b^(d/2) = environ 7^2 = 49x de reduction. L'ordonnancement actuel (par proximite au centre) est heuristique mais non optimal. Pour ameliorer, on pourrait : (1) trier par valeur de l'itération précédente, (2) utiliser une table de transposition, (3) ordonnancer par les coups les plus joues. Le **speedup wall-clock** (rapport des temps d'exécution) est en revanche *machine-dep* et depend du runtime, de la charge système et de la taille de l'instance ; il **ne survit pas** a une re-exécution sur une autre machine, même si l'ordre de grandeur est preserve. Pour observer vos propres timings, executez la cellule de benchmark ci-dessous.



## 5. Algorithme MCTS

### Principe de Monte Carlo Tree Search

MCTS construit progressivement un arbre de recherche en equilibant exploration et exploitation :

1. **sélection** : Descendre dans l'arbre avec UCB1
2. **Expansion** : Ajouter un nouveau noeud
3. **Simulation** : Jouer une partie aléatoire (rollout)
4. **Backpropagation** : Remonter le résultat

### Formule UCB1

$$UCB1 = \frac{W}{N} + c \sqrt{\frac{\ln(N_{parent})}{N}}$$

Ou :
- W = nombre de victoires
- N = nombre de visites
- c = paramètre d'exploration (typiquement 1.41)

In [7]:
// Moteur de jeu Connect Four : grille de 6 lignes x 7 colonnes.
// 0 = case vide, 1 = joueur MAX (X), -1 = joueur MIN (O).
#[allow(dead_code)]
#[derive(Clone)]
struct Board {
    grid: [[i32; 7]; 6],
    current_player: i32,
}

#[allow(dead_code)]
impl Board {
    fn new() -> Board {
        Board { grid: [[0; 7]; 6], current_player: 1 }
    }
    fn state(&self) -> Board {
        self.clone()
    }
    fn set_state(&mut self, s: Board) {
        self.grid = s.grid;
        self.current_player = s.current_player;
    }
    // Colonnes dont la case du haut est libre.
    fn valid_moves(&self) -> Vec<i32> {
        let mut moves = Vec::new();
        for c in 0..7 {
            if self.grid[0][c] == 0 {
                moves.push(c as i32);
            }
        }
        moves
    }
    // Pose le jeton par gravite ; le joueur courant change APRES la pose.
    fn drop_piece(&mut self, col: i32) -> bool {
        if col < 0 || col >= 7 {
            return false;
        }
        let c = col as usize;
        if self.grid[0][c] != 0 {
            return false;
        }
        for row in (0..6).rev() {
            if self.grid[row][c] == 0 {
                self.grid[row][c] = self.current_player;
                self.current_player *= -1;
                return true;
            }
        }
        false
    }
    // Retire le jeton du haut de la colonne ; le joueur courant change aussi.
    fn undo_move(&mut self, col: i32) {
        let c = col as usize;
        for row in 0..6 {
            if self.grid[row][c] != 0 {
                self.grid[row][c] = 0;
                self.current_player *= -1;
                return;
            }
        }
    }
    // Ordre des directions : horizontales, verticales, diagonales bas-droite,
    // diagonales bas-gauche.
    fn check_winner(&self) -> i32 {
        for r in 0..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r][c + 1], self.grid[r][c + 2], self.grid[r][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..7 {
                let w = [self.grid[r][c], self.grid[r + 1][c], self.grid[r + 2][c], self.grid[r + 3][c]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r + 1][c + 1], self.grid[r + 2][c + 2], self.grid[r + 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 3..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r - 1][c + 1], self.grid[r - 2][c + 2], self.grid[r - 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        0
    }
    fn is_terminal(&self) -> bool {
        self.check_winner() != 0 || self.valid_moves().is_empty()
    }
    fn get_utility(&self, maximizing_player: i32) -> f64 {
        (self.check_winner() * maximizing_player) as f64
    }
    fn display(&self) -> String {
        let sym = |c: i32| -> char {
            if c == 0 {
                '.'
            } else if c == 1 {
                'X'
            } else {
                'O'
            }
        };
        let mut lines: Vec<String> = Vec::new();
        for r in 0..6 {
            let cells: Vec<String> = (0..7).map(|c| sym(self.grid[r][c]).to_string()).collect();
            lines.push(cells.join(" "));
        }
        let cols: Vec<String> = (0..7).map(|c| c.to_string()).collect();
        lines.push(cols.join(" "));
        lines.join("\n")
    }
}
// Heuristique : score des fenetres de 4 cases + controle du centre.
#[allow(dead_code)]
fn evaluate_window(window: &[i32; 4], player: i32) -> i64 {
    let mut score: i64 = 0;
    let opponent = -player;

    let player_count = window.iter().filter(|&&x| x == player).count();
    let opponent_count = window.iter().filter(|&&x| x == opponent).count();
    let empty_count = window.iter().filter(|&&x| x == 0).count();

    // Fenetre gagnante
    if player_count == 4 {
        score += 100;
    } else if player_count == 3 && empty_count == 1 {
        score += 5;
    } else if player_count == 2 && empty_count == 2 {
        score += 2;
    }

    // Bloquer l'adversaire (3 jetons adverses + 1 vide)
    if opponent_count == 3 && empty_count == 1 {
        score -= 4;
    }

    score
}

#[allow(dead_code)]
fn evaluate_position(game: &Board, player: i32) -> f64 {
    if game.is_terminal() {
        return game.get_utility(player) * 1000.0;
    }

    let mut score: i64 = 0;

    // Bonus pour le centre (colonne 3)
    for r in 0..6 {
        if game.grid[r][3] == player {
            score += 3;
        }
    }

    // Horizontales
    for r in 0..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r][c + 1], game.grid[r][c + 2], game.grid[r][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Verticales
    for r in 0..3 {
        for c in 0..7 {
            let w = [game.grid[r][c], game.grid[r + 1][c], game.grid[r + 2][c], game.grid[r + 3][c]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-droite
    for r in 0..3 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r + 1][c + 1], game.grid[r + 2][c + 2], game.grid[r + 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-gauche
    for r in 3..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r - 1][c + 1], game.grid[r - 2][c + 2], game.grid[r - 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    score as f64
}
// Statistiques de recherche et Minimax (sans tri des coups).
#[allow(dead_code)]
#[derive(Default)]
struct Stats {
    nodes: u64,
}

// Minimax : les coups sont parcourus dans l'ordre brut des colonnes,
// et le premier maximum strict est retenu.
#[allow(dead_code)]
fn minimax(
    game: &mut Board,
    depth: i32,
    maximizing: bool,
    stats: &mut Stats,
    player: i32,
) -> (f64, Option<i32>) {
    stats.nodes += 1;

    if depth == 0 || game.is_terminal() {
        if game.is_terminal() {
            return (game.get_utility(player) * 1000.0, None);
        }
        return (evaluate_position(game, player), None);
    }

    let valid_moves = game.valid_moves();

    if maximizing {
        let mut best_value = f64::NEG_INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, false, stats, player);
            game.undo_move(mv);
            if value > best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    } else {
        let mut best_value = f64::INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, true, stats, player);
            game.undo_move(mv);
            if value < best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    }
}
// MT19937 compatible avec random.Random de CPython (init_by_array).
// Le carnet sème 42 avant chaque demonstration stochastique : c'est ce
// generateur qui rend les sorties reproductibles.
#[allow(dead_code)]
struct Mt19937 {
    mt: [u32; 624],
    idx: usize,
}

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32
                .wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30))
                .wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 {
        Mt19937::new_by_array(&[seed])
    }
    fn new_by_array(key: &[u32]) -> Mt19937 {
        let mut mt = Mt19937::init_genrand(19650218);
        let mut i: usize = 1;
        let mut j: usize = 0;
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j])
                .wrapping_add(j as u32);
            i += 1;
            j += 1;
            if i >= 624 {
                mt[0] = mt[623];
                i = 1;
            }
            if j >= key.len() {
                j = 0;
            }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 {
                mt[0] = mt[623];
                i = 1;
            }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 {
                    v ^= 0x9908_b0df;
                }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn getrandbits(&mut self, k: u32) -> u64 {
        if k <= 32 {
            return (self.next_u32() >> (32 - k)) as u64;
        }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    // Rejet, comme _randbelow de CPython.
    fn randbelow(&mut self, n: u64) -> u64 {
        if n == 0 {
            return 0;
        }
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n {
            r = self.getrandbits(k);
        }
        r
    }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T {
        &seq[self.randbelow(seq.len() as u64) as usize]
    }
}
// Monte Carlo Tree Search. L'arbre est un arena : chaque noeud porte son
// parent, ses enfants (dans l'ordre d'insertion) et ses compteurs.
#[allow(dead_code)]
#[derive(Clone)]
struct MctsNode {
    state: Board,
    action: Option<i32>,
    parent: Option<usize>,
    children: Vec<usize>,
    visits: u64,
    wins: f64,
    untried: Option<Vec<i32>>,
}

#[allow(dead_code)]
struct Mcts {
    c: f64,
    stats: Stats,
}

// Score UCB1 : infini tant que le noeud n'a pas ete visite.
#[allow(dead_code)]
fn ucb1(arena: &[MctsNode], index: usize, c: f64) -> f64 {
    let node = &arena[index];
    if node.visits == 0 {
        return f64::INFINITY;
    }
    let exploitation = node.wins / node.visits as f64;
    let parent_visits = match node.parent {
        Some(p) => arena[p].visits,
        None => 0,
    };
    if parent_visits == 0 {
        return f64::INFINITY;
    }
    let exploration = c * ((parent_visits as f64).ln() / node.visits as f64).sqrt();
    exploitation + exploration
}

// Premier maximum strict (comme max() de Python sur un dict ordonne).
#[allow(dead_code)]
fn best_child_ucb1(arena: &[MctsNode], index: usize, c: f64) -> usize {
    let mut best = arena[index].children[0];
    let mut best_score = ucb1(arena, best, c);
    for &child in arena[index].children.iter().skip(1) {
        let score = ucb1(arena, child, c);
        if score > best_score {
            best_score = score;
            best = child;
        }
    }
    best
}

// Premier maximum strict du nombre de visites (coup final).
#[allow(dead_code)]
fn best_child_visits(arena: &[MctsNode], index: usize) -> usize {
    let mut best = arena[index].children[0];
    let mut best_visits = arena[best].visits;
    for &child in arena[index].children.iter().skip(1) {
        if arena[child].visits > best_visits {
            best_visits = arena[child].visits;
            best = child;
        }
    }
    best
}

// Selection : descente UCB1 jusqu'a un noeud terminal ou non pleinement expandu.
#[allow(dead_code)]
fn select(
    arena: &mut Vec<MctsNode>,
    mut index: usize,
    game: &mut Board,
    c: f64,
    stats: &mut Stats,
) -> usize {
    loop {
        let state = arena[index].state.clone();
        game.set_state(state);
        if game.is_terminal() {
            return index;
        }
        if arena[index].untried.is_none() {
            let state = arena[index].state.clone();
            game.set_state(state);
            arena[index].untried = Some(game.valid_moves());
        }
        let untried_empty = arena[index].untried.as_ref().map(|v| v.is_empty()).unwrap_or(false);
        let has_children = !arena[index].children.is_empty();
        if !(untried_empty && has_children) {
            return expand(arena, index, game, stats);
        }
        let best = best_child_ucb1(arena, index, c);
        index = best;
        let state = arena[index].state.clone();
        game.set_state(state);
    }
}

// Expansion : le dernier coup restant est pris (LIFO), sans tirage.
#[allow(dead_code)]
fn expand(
    arena: &mut Vec<MctsNode>,
    index: usize,
    game: &mut Board,
    stats: &mut Stats,
) -> usize {
    stats.nodes += 1;
    if arena[index].untried.is_none() {
        let state = arena[index].state.clone();
        game.set_state(state);
        arena[index].untried = Some(game.valid_moves());
    }
    let action = match arena[index].untried.as_mut() {
        Some(list) => list.pop(),
        None => None,
    };
    let action = match action {
        Some(a) => a,
        None => return index,
    };
    let state = arena[index].state.clone();
    game.set_state(state);
    game.drop_piece(action);
    let child_state = game.state();
    arena.push(MctsNode {
        state: child_state,
        action: Some(action),
        parent: Some(index),
        children: Vec::new(),
        visits: 0,
        wins: 0.0,
        untried: None,
    });
    let child = arena.len() - 1;
    arena[index].children.push(child);
    child
}

// Simulation : partie aleatoire jusqu'a un etat terminal.
#[allow(dead_code)]
fn simulate(game: &mut Board, rng: &mut Mt19937, stats: &mut Stats) -> f64 {
    stats.nodes += 1;
    let saved = game.state();
    let original_player = game.current_player;
    while !game.is_terminal() {
        let moves = game.valid_moves();
        if moves.is_empty() {
            break;
        }
        let mv = *rng.choice(&moves);
        game.drop_piece(mv);
    }
    let result = game.get_utility(original_player);
    game.set_state(saved);
    result
}

// Backpropagation : le resultat est inverse selon le joueur du noeud.
#[allow(dead_code)]
fn backpropagate(arena: &mut Vec<MctsNode>, mut index: usize, result: f64) {
    loop {
        arena[index].visits += 1;
        let player = arena[index].state.current_player;
        if player == -1 {
            arena[index].wins += result;
        } else {
            arena[index].wins -= result;
        }
        match arena[index].parent {
            Some(parent) => index = parent,
            None => break,
        }
    }
}

#[allow(dead_code)]
impl Mcts {
    // L'etat du jeu du caller n'est jamais modifie : il est sauvegarde en
    // entree et restaure en sortie.
    fn search(
        &mut self,
        game: &mut Board,
        iterations: u32,
        rng: &mut Mt19937,
    ) -> (Option<i32>, f64) {
        let c = self.c;
        let mut stats = Stats { nodes: 0 };
        let initial = game.state();
        let mut arena: Vec<MctsNode> = Vec::new();
        arena.push(MctsNode {
            state: initial.clone(),
            action: None,
            parent: None,
            children: Vec::new(),
            visits: 0,
            wins: 0.0,
            untried: None,
        });

        for _ in 0..iterations {
            let node = select(&mut arena, 0, game, c, &mut stats);
            let result = simulate(game, rng, &mut stats);
            backpropagate(&mut arena, node, result);
        }

        game.set_state(initial);
        self.stats = stats;

        if arena[0].children.is_empty() {
            return (None, 0.0);
        }
        let best = best_child_visits(&arena, 0);
        let visits = arena[best].visits;
        let win_rate = if visits > 0 {
            arena[best].wins / visits as f64
        } else {
            0.0
        };
        (arena[best].action, win_rate)
    }
}
// Separateur de milliers, comme le format `:,` de Python.
#[allow(dead_code)]
fn thousands(n: u64) -> String {
    let s = n.to_string();
    let len = s.len();
    let mut out = String::new();
    for (i, ch) in s.chars().enumerate() {
        if i > 0 && (len - i) % 3 == 0 {
            out.push(',');
        }
        out.push(ch);
    }
    out
}

// Liste d'entiers au format de la repr() d'une liste Python.
#[allow(dead_code)]
fn py_list_i32(values: &[i32]) -> String {
    let items: Vec<String> = values.iter().map(|v| v.to_string()).collect();
    format!("[{}]", items.join(", "))
}

// Un Option imprime comme un Optional Python.
#[allow(dead_code)]
fn py_opt_i32(value: Option<i32>) -> String {
    match value {
        Some(v) => v.to_string(),
        None => "None".to_string(),
    }
}
// Graine posee avant la demonstration : les rollouts sont aleatoires, sans
// elle le taux affiche changerait a chaque execution.
let mut rng = Mt19937::new(42);
let mut game = Board::new();
let mut mcts = Mcts { c: 1.41, stats: Stats { nodes: 0 } };

let start = std::time::Instant::now();
let (mv, win_rate) = mcts.search(&mut game, 1000, &mut rng);
let elapsed = start.elapsed().as_secs_f64();

println!("MCTS (1000 iterations):");
println!("  Meilleur coup: {}", py_opt_i32(mv));
println!("  Taux de victoire estime: {:.3}", win_rate);
println!("  Noeuds explores: {}", thousands(mcts.stats.nodes));
println!("  Temps: {:.4}s", elapsed);


MCTS (1000 iterations):


  Meilleur coup: 5


  Taux de victoire estime: 0.109


  Noeuds explores: 2,000


  Temps: 0.0066s


### Interprétation : Test MCTS

**Sortie obtenue** : MCTS avec 1000 itérations sur position initiale (graine posee dans la cellule de test : sortie reproductible).

| Métrique | Valeur | Analyse |
|----------|--------|---------|
| Meilleur coup | 5 | Colonne périphérique -- le MCTS naïf à ce budget ne trouve pas le centre |
| Taux de victoire | 0.109 | Valeur faible et bruitée pour 1000 itérations (convention détaillée dans le benchmark, section 6) |
| Noeuds explores | 2,000 | 1000 selections + 1000 simulations (invariant structurel sur ce budget) |
| Temps | Runtime *machine-dep* | Coût wall-clock pour 1000 itérations (Rust, noyau evcxr) |

**Points clés** :
1. **Pas de coup central** : contrairement à Minimax et Alpha-Beta, le MCTS à rollouts uniformes ne privilégie pas la colonne 3 à ce budget -- le signal des parties aléatoires est trop pauvre pour discriminer les colonnes sur Connect Four
2. **Taux de victoire modere** : 0.109, estimation bruitée pour 1000 itérations
3. **convergence lente** : MCTS nécessite beaucoup plus d'itérations (ou des rollout policies biaisées) pour affiner Connect Four -- le tournoi de la section suivante le confirme
4. **Speedup wall-clock** : ratio MCTS/Alpha-Beta = *machine-dep* (Rust, noyau evcxr + budget 1000 iter)

> **Note technique** : MCTS est **probabiliste** : sans graine, chaque exécution donne un résultat legerement différent -- la graine posee dans la cellule de test rend cette sortie précise reproductible. Le **taux de victoire** estime reste faible car les rollouts aléatoires exploitent mal la structure tactique forte de Connect Four. Le **budget d'itérations** (1000) et le **nombre de noeuds explores** (2,000 = selections + simulations) sont des invariants structurels du budget de simulation (déterministes sur le budget) ; le **temps d'exécution** est en revanche *machine-dep* (Rust, noyau evcxr + charge + 1000 itérations). MCTS brille sur les jeux a grand facteur de branchement (Go, Havannah) ou les rollouts explorent des espaces vastes. Pour Connect Four, Alpha-Beta avec une bonne heuristique est nettement superieur. Pour observer vos propres timings, executez la cellule de benchmark ci-dessous.

## 6. Benchmark Comparatif

Comparons les trois algorithmes sur différentes dimensions :
- **Temps d'exécution**
- **Noeuds explores**
- **Qualité du coup** (vs coup optimal)

In [8]:
// Moteur de jeu Connect Four : grille de 6 lignes x 7 colonnes.
// 0 = case vide, 1 = joueur MAX (X), -1 = joueur MIN (O).
#[allow(dead_code)]
#[derive(Clone)]
struct Board {
    grid: [[i32; 7]; 6],
    current_player: i32,
}

#[allow(dead_code)]
impl Board {
    fn new() -> Board {
        Board { grid: [[0; 7]; 6], current_player: 1 }
    }
    fn state(&self) -> Board {
        self.clone()
    }
    fn set_state(&mut self, s: Board) {
        self.grid = s.grid;
        self.current_player = s.current_player;
    }
    // Colonnes dont la case du haut est libre.
    fn valid_moves(&self) -> Vec<i32> {
        let mut moves = Vec::new();
        for c in 0..7 {
            if self.grid[0][c] == 0 {
                moves.push(c as i32);
            }
        }
        moves
    }
    // Pose le jeton par gravite ; le joueur courant change APRES la pose.
    fn drop_piece(&mut self, col: i32) -> bool {
        if col < 0 || col >= 7 {
            return false;
        }
        let c = col as usize;
        if self.grid[0][c] != 0 {
            return false;
        }
        for row in (0..6).rev() {
            if self.grid[row][c] == 0 {
                self.grid[row][c] = self.current_player;
                self.current_player *= -1;
                return true;
            }
        }
        false
    }
    // Retire le jeton du haut de la colonne ; le joueur courant change aussi.
    fn undo_move(&mut self, col: i32) {
        let c = col as usize;
        for row in 0..6 {
            if self.grid[row][c] != 0 {
                self.grid[row][c] = 0;
                self.current_player *= -1;
                return;
            }
        }
    }
    // Ordre des directions : horizontales, verticales, diagonales bas-droite,
    // diagonales bas-gauche.
    fn check_winner(&self) -> i32 {
        for r in 0..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r][c + 1], self.grid[r][c + 2], self.grid[r][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..7 {
                let w = [self.grid[r][c], self.grid[r + 1][c], self.grid[r + 2][c], self.grid[r + 3][c]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r + 1][c + 1], self.grid[r + 2][c + 2], self.grid[r + 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 3..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r - 1][c + 1], self.grid[r - 2][c + 2], self.grid[r - 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        0
    }
    fn is_terminal(&self) -> bool {
        self.check_winner() != 0 || self.valid_moves().is_empty()
    }
    fn get_utility(&self, maximizing_player: i32) -> f64 {
        (self.check_winner() * maximizing_player) as f64
    }
    fn display(&self) -> String {
        let sym = |c: i32| -> char {
            if c == 0 {
                '.'
            } else if c == 1 {
                'X'
            } else {
                'O'
            }
        };
        let mut lines: Vec<String> = Vec::new();
        for r in 0..6 {
            let cells: Vec<String> = (0..7).map(|c| sym(self.grid[r][c]).to_string()).collect();
            lines.push(cells.join(" "));
        }
        let cols: Vec<String> = (0..7).map(|c| c.to_string()).collect();
        lines.push(cols.join(" "));
        lines.join("\n")
    }
}
// Heuristique : score des fenetres de 4 cases + controle du centre.
#[allow(dead_code)]
fn evaluate_window(window: &[i32; 4], player: i32) -> i64 {
    let mut score: i64 = 0;
    let opponent = -player;

    let player_count = window.iter().filter(|&&x| x == player).count();
    let opponent_count = window.iter().filter(|&&x| x == opponent).count();
    let empty_count = window.iter().filter(|&&x| x == 0).count();

    // Fenetre gagnante
    if player_count == 4 {
        score += 100;
    } else if player_count == 3 && empty_count == 1 {
        score += 5;
    } else if player_count == 2 && empty_count == 2 {
        score += 2;
    }

    // Bloquer l'adversaire (3 jetons adverses + 1 vide)
    if opponent_count == 3 && empty_count == 1 {
        score -= 4;
    }

    score
}

#[allow(dead_code)]
fn evaluate_position(game: &Board, player: i32) -> f64 {
    if game.is_terminal() {
        return game.get_utility(player) * 1000.0;
    }

    let mut score: i64 = 0;

    // Bonus pour le centre (colonne 3)
    for r in 0..6 {
        if game.grid[r][3] == player {
            score += 3;
        }
    }

    // Horizontales
    for r in 0..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r][c + 1], game.grid[r][c + 2], game.grid[r][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Verticales
    for r in 0..3 {
        for c in 0..7 {
            let w = [game.grid[r][c], game.grid[r + 1][c], game.grid[r + 2][c], game.grid[r + 3][c]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-droite
    for r in 0..3 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r + 1][c + 1], game.grid[r + 2][c + 2], game.grid[r + 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-gauche
    for r in 3..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r - 1][c + 1], game.grid[r - 2][c + 2], game.grid[r - 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    score as f64
}
// Statistiques de recherche et Minimax (sans tri des coups).
#[allow(dead_code)]
#[derive(Default)]
struct Stats {
    nodes: u64,
}

// Minimax : les coups sont parcourus dans l'ordre brut des colonnes,
// et le premier maximum strict est retenu.
#[allow(dead_code)]
fn minimax(
    game: &mut Board,
    depth: i32,
    maximizing: bool,
    stats: &mut Stats,
    player: i32,
) -> (f64, Option<i32>) {
    stats.nodes += 1;

    if depth == 0 || game.is_terminal() {
        if game.is_terminal() {
            return (game.get_utility(player) * 1000.0, None);
        }
        return (evaluate_position(game, player), None);
    }

    let valid_moves = game.valid_moves();

    if maximizing {
        let mut best_value = f64::NEG_INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, false, stats, player);
            game.undo_move(mv);
            if value > best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    } else {
        let mut best_value = f64::INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, true, stats, player);
            game.undo_move(mv);
            if value < best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    }
}
// Alpha-Beta : les coups sont ordonnes par proximite au centre (tri stable),
// et la coupe s'applique quand beta <= alpha.
#[allow(dead_code)]
fn alpha_beta(
    game: &mut Board,
    depth: i32,
    mut alpha: f64,
    mut beta: f64,
    maximizing: bool,
    stats: &mut Stats,
    player: i32,
) -> (f64, Option<i32>) {
    stats.nodes += 1;

    if depth == 0 || game.is_terminal() {
        if game.is_terminal() {
            return (game.get_utility(player) * 1000.0, None);
        }
        return (evaluate_position(game, player), None);
    }

    let mut valid_moves = game.valid_moves();
    valid_moves.sort_by_key(|&c| (c - 3).abs());

    if maximizing {
        let mut best_value = f64::NEG_INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = alpha_beta(game, depth - 1, alpha, beta, false, stats, player);
            game.undo_move(mv);
            if value > best_value {
                best_value = value;
                best_move = Some(mv);
            }
            if best_value > alpha {
                alpha = best_value;
            }
            if beta <= alpha {
                break;
            }
        }
        (best_value, best_move)
    } else {
        let mut best_value = f64::INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = alpha_beta(game, depth - 1, alpha, beta, true, stats, player);
            game.undo_move(mv);
            if value < best_value {
                best_value = value;
                best_move = Some(mv);
            }
            if best_value < beta {
                beta = best_value;
            }
            if beta <= alpha {
                break;
            }
        }
        (best_value, best_move)
    }
}
let mut game = Board::new();

println!("Benchmark par profondeur:");
{
    for depth in 1..=6 {
        let mut stats_mm = Stats { nodes: 0 };
        let start_mm = std::time::Instant::now();
        let _ = minimax(&mut game, depth, true, &mut stats_mm, 1);
        let time_mm = start_mm.elapsed().as_secs_f64();

        let mut stats_ab = Stats { nodes: 0 };
        let start_ab = std::time::Instant::now();
        let _ = alpha_beta(
            &mut game,
            depth,
            f64::NEG_INFINITY,
            f64::INFINITY,
            true,
            &mut stats_ab,
            1,
        );
        let time_ab = start_ab.elapsed().as_secs_f64();

        let ratio = stats_mm.nodes as f64 / stats_ab.nodes.max(1) as f64;
        println!(
            "d={}: Minimax {} noeuds {:.4}s, Alpha-Beta {} noeuds {:.4}s, speedup noeuds {:.6}",
            depth, stats_mm.nodes, time_mm, stats_ab.nodes, time_ab, ratio
        );
    }
};


Benchmark par profondeur:


d=1: Minimax 8 noeuds 0.0000s, Alpha-Beta 8 noeuds 0.0000s, speedup noeuds 1.000000


d=2: Minimax 57 noeuds 0.0000s, Alpha-Beta 21 noeuds 0.0000s, speedup noeuds 2.714286


d=3: Minimax 400 noeuds 0.0003s, Alpha-Beta 82 noeuds 0.0000s, speedup noeuds 4.878049


d=4: Minimax 2801 noeuds 0.0016s, Alpha-Beta 178 noeuds 0.0001s, speedup noeuds 15.735955


d=5: Minimax 19608 noeuds 0.0108s, Alpha-Beta 641 noeuds 0.0004s, speedup noeuds 30.589704


d=6: Minimax 137257 noeuds 0.1056s, Alpha-Beta 1370 noeuds 0.0006s, speedup noeuds 100.187591


### Interprétation : Benchmark par profondeur

**Sortie obtenue** : Comparaison Minimax vs Alpha-Beta de profondeur 1 a 6.

| Profondeur | Minimax noeuds | Alpha-Beta noeuds | Speedup noeuds | Minimax temps | Alpha-Beta temps | Speedup temps |
|------------|----------------|-------------------|----------------|---------------|------------------|---------------|
| 1 | 8 | 8 | 1.0x | Runtime *machine-dep* | Runtime *machine-dep* | Speedup wall-clock *machine-dep* |
| 2 | 57 | 21 | 2.7x | Runtime *machine-dep* | Runtime *machine-dep* | Speedup wall-clock *machine-dep* |
| 3 | 400 | 82 | 4.9x | Runtime *machine-dep* | Runtime *machine-dep* | Speedup wall-clock *machine-dep* |
| 4 | 2,801 | 178 | 15.7x | Runtime *machine-dep* | Runtime *machine-dep* | Speedup wall-clock *machine-dep* |
| 5 | 19,608 | 641 | 30.6x | Runtime *machine-dep* | Runtime *machine-dep* | Speedup wall-clock *machine-dep* |
| 6 | 137,257 | 1,370 | 100.2x | Runtime *machine-dep* | Runtime *machine-dep* | Speedup wall-clock *machine-dep* |

**Points clés** :
1. **Speedup noeuds exponentiel** : Le **speedup structurel** (rapport des **noeuds explores**, invariant algorithmique déterministe) augmente de 1x a 100x entre d=1 et d=6 — preuve que l'elagage Alpha-Beta est d'autant plus efficace que la profondeur augmente
2. **Complexité Minimax** : 137k noeuds a d=6 = explosion combinatoire (b^d avec b~4-7)
3. **Efficacite Alpha-Beta** : Seulement 1.4k noeuds a d=6 = elagage massif (invariant structurel)
4. **Speedup temps wall-clock** : le **speedup runtime** (rapport des temps d'exécution) depend du runtime, de la charge système et de la profondeur ; il **ne survit pas** a une re-exécution

> **Note methodologique -- separation structurel / machine-dep** : le theoreme dit qu'Alpha-Beta est O(b^(d/2)) avec ordonnancement optimal vs O(b^d) pour Minimax. Sur Connect Four 7x6, le **nombre de noeuds explores** (Minimax 137,257 / Alpha-Beta 1,370 a d=6) est un **invariant structurel** du problème sur l'instance (déterministe, propriete algorithmique du récursive search sans randomness). En revanche, le **temps d'exécution** et le **speedup wall-clock** dependent du runtime, de la charge système et de la taille de l'instance -- ils **ne survivent pas** a une re-exécution sur une autre machine, même si l'ordre de grandeur est preserve. Les durées exactes sont celles lues dans la sortie du benchmark ci-dessus. Les valeurs observees sur cette machine sont conformes au pattern théorique b^(d/2) ; pour observer vos propres timings, executez la cellule de benchmark ci-dessous.



Benchmark MCTS avec différents nombres d'itérations.

In [9]:
// Moteur de jeu Connect Four : grille de 6 lignes x 7 colonnes.
// 0 = case vide, 1 = joueur MAX (X), -1 = joueur MIN (O).
#[allow(dead_code)]
#[derive(Clone)]
struct Board {
    grid: [[i32; 7]; 6],
    current_player: i32,
}

#[allow(dead_code)]
impl Board {
    fn new() -> Board {
        Board { grid: [[0; 7]; 6], current_player: 1 }
    }
    fn state(&self) -> Board {
        self.clone()
    }
    fn set_state(&mut self, s: Board) {
        self.grid = s.grid;
        self.current_player = s.current_player;
    }
    // Colonnes dont la case du haut est libre.
    fn valid_moves(&self) -> Vec<i32> {
        let mut moves = Vec::new();
        for c in 0..7 {
            if self.grid[0][c] == 0 {
                moves.push(c as i32);
            }
        }
        moves
    }
    // Pose le jeton par gravite ; le joueur courant change APRES la pose.
    fn drop_piece(&mut self, col: i32) -> bool {
        if col < 0 || col >= 7 {
            return false;
        }
        let c = col as usize;
        if self.grid[0][c] != 0 {
            return false;
        }
        for row in (0..6).rev() {
            if self.grid[row][c] == 0 {
                self.grid[row][c] = self.current_player;
                self.current_player *= -1;
                return true;
            }
        }
        false
    }
    // Retire le jeton du haut de la colonne ; le joueur courant change aussi.
    fn undo_move(&mut self, col: i32) {
        let c = col as usize;
        for row in 0..6 {
            if self.grid[row][c] != 0 {
                self.grid[row][c] = 0;
                self.current_player *= -1;
                return;
            }
        }
    }
    // Ordre des directions : horizontales, verticales, diagonales bas-droite,
    // diagonales bas-gauche.
    fn check_winner(&self) -> i32 {
        for r in 0..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r][c + 1], self.grid[r][c + 2], self.grid[r][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..7 {
                let w = [self.grid[r][c], self.grid[r + 1][c], self.grid[r + 2][c], self.grid[r + 3][c]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r + 1][c + 1], self.grid[r + 2][c + 2], self.grid[r + 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 3..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r - 1][c + 1], self.grid[r - 2][c + 2], self.grid[r - 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        0
    }
    fn is_terminal(&self) -> bool {
        self.check_winner() != 0 || self.valid_moves().is_empty()
    }
    fn get_utility(&self, maximizing_player: i32) -> f64 {
        (self.check_winner() * maximizing_player) as f64
    }
    fn display(&self) -> String {
        let sym = |c: i32| -> char {
            if c == 0 {
                '.'
            } else if c == 1 {
                'X'
            } else {
                'O'
            }
        };
        let mut lines: Vec<String> = Vec::new();
        for r in 0..6 {
            let cells: Vec<String> = (0..7).map(|c| sym(self.grid[r][c]).to_string()).collect();
            lines.push(cells.join(" "));
        }
        let cols: Vec<String> = (0..7).map(|c| c.to_string()).collect();
        lines.push(cols.join(" "));
        lines.join("\n")
    }
}
// Heuristique : score des fenetres de 4 cases + controle du centre.
#[allow(dead_code)]
fn evaluate_window(window: &[i32; 4], player: i32) -> i64 {
    let mut score: i64 = 0;
    let opponent = -player;

    let player_count = window.iter().filter(|&&x| x == player).count();
    let opponent_count = window.iter().filter(|&&x| x == opponent).count();
    let empty_count = window.iter().filter(|&&x| x == 0).count();

    // Fenetre gagnante
    if player_count == 4 {
        score += 100;
    } else if player_count == 3 && empty_count == 1 {
        score += 5;
    } else if player_count == 2 && empty_count == 2 {
        score += 2;
    }

    // Bloquer l'adversaire (3 jetons adverses + 1 vide)
    if opponent_count == 3 && empty_count == 1 {
        score -= 4;
    }

    score
}

#[allow(dead_code)]
fn evaluate_position(game: &Board, player: i32) -> f64 {
    if game.is_terminal() {
        return game.get_utility(player) * 1000.0;
    }

    let mut score: i64 = 0;

    // Bonus pour le centre (colonne 3)
    for r in 0..6 {
        if game.grid[r][3] == player {
            score += 3;
        }
    }

    // Horizontales
    for r in 0..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r][c + 1], game.grid[r][c + 2], game.grid[r][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Verticales
    for r in 0..3 {
        for c in 0..7 {
            let w = [game.grid[r][c], game.grid[r + 1][c], game.grid[r + 2][c], game.grid[r + 3][c]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-droite
    for r in 0..3 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r + 1][c + 1], game.grid[r + 2][c + 2], game.grid[r + 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-gauche
    for r in 3..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r - 1][c + 1], game.grid[r - 2][c + 2], game.grid[r - 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    score as f64
}
// Statistiques de recherche et Minimax (sans tri des coups).
#[allow(dead_code)]
#[derive(Default)]
struct Stats {
    nodes: u64,
}

// Minimax : les coups sont parcourus dans l'ordre brut des colonnes,
// et le premier maximum strict est retenu.
#[allow(dead_code)]
fn minimax(
    game: &mut Board,
    depth: i32,
    maximizing: bool,
    stats: &mut Stats,
    player: i32,
) -> (f64, Option<i32>) {
    stats.nodes += 1;

    if depth == 0 || game.is_terminal() {
        if game.is_terminal() {
            return (game.get_utility(player) * 1000.0, None);
        }
        return (evaluate_position(game, player), None);
    }

    let valid_moves = game.valid_moves();

    if maximizing {
        let mut best_value = f64::NEG_INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, false, stats, player);
            game.undo_move(mv);
            if value > best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    } else {
        let mut best_value = f64::INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, true, stats, player);
            game.undo_move(mv);
            if value < best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    }
}
// MT19937 compatible avec random.Random de CPython (init_by_array).
// Le carnet sème 42 avant chaque demonstration stochastique : c'est ce
// generateur qui rend les sorties reproductibles.
#[allow(dead_code)]
struct Mt19937 {
    mt: [u32; 624],
    idx: usize,
}

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32
                .wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30))
                .wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 {
        Mt19937::new_by_array(&[seed])
    }
    fn new_by_array(key: &[u32]) -> Mt19937 {
        let mut mt = Mt19937::init_genrand(19650218);
        let mut i: usize = 1;
        let mut j: usize = 0;
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j])
                .wrapping_add(j as u32);
            i += 1;
            j += 1;
            if i >= 624 {
                mt[0] = mt[623];
                i = 1;
            }
            if j >= key.len() {
                j = 0;
            }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 {
                mt[0] = mt[623];
                i = 1;
            }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 {
                    v ^= 0x9908_b0df;
                }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn getrandbits(&mut self, k: u32) -> u64 {
        if k <= 32 {
            return (self.next_u32() >> (32 - k)) as u64;
        }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    // Rejet, comme _randbelow de CPython.
    fn randbelow(&mut self, n: u64) -> u64 {
        if n == 0 {
            return 0;
        }
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n {
            r = self.getrandbits(k);
        }
        r
    }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T {
        &seq[self.randbelow(seq.len() as u64) as usize]
    }
}
// Monte Carlo Tree Search. L'arbre est un arena : chaque noeud porte son
// parent, ses enfants (dans l'ordre d'insertion) et ses compteurs.
#[allow(dead_code)]
#[derive(Clone)]
struct MctsNode {
    state: Board,
    action: Option<i32>,
    parent: Option<usize>,
    children: Vec<usize>,
    visits: u64,
    wins: f64,
    untried: Option<Vec<i32>>,
}

#[allow(dead_code)]
struct Mcts {
    c: f64,
    stats: Stats,
}

// Score UCB1 : infini tant que le noeud n'a pas ete visite.
#[allow(dead_code)]
fn ucb1(arena: &[MctsNode], index: usize, c: f64) -> f64 {
    let node = &arena[index];
    if node.visits == 0 {
        return f64::INFINITY;
    }
    let exploitation = node.wins / node.visits as f64;
    let parent_visits = match node.parent {
        Some(p) => arena[p].visits,
        None => 0,
    };
    if parent_visits == 0 {
        return f64::INFINITY;
    }
    let exploration = c * ((parent_visits as f64).ln() / node.visits as f64).sqrt();
    exploitation + exploration
}

// Premier maximum strict (comme max() de Python sur un dict ordonne).
#[allow(dead_code)]
fn best_child_ucb1(arena: &[MctsNode], index: usize, c: f64) -> usize {
    let mut best = arena[index].children[0];
    let mut best_score = ucb1(arena, best, c);
    for &child in arena[index].children.iter().skip(1) {
        let score = ucb1(arena, child, c);
        if score > best_score {
            best_score = score;
            best = child;
        }
    }
    best
}

// Premier maximum strict du nombre de visites (coup final).
#[allow(dead_code)]
fn best_child_visits(arena: &[MctsNode], index: usize) -> usize {
    let mut best = arena[index].children[0];
    let mut best_visits = arena[best].visits;
    for &child in arena[index].children.iter().skip(1) {
        if arena[child].visits > best_visits {
            best_visits = arena[child].visits;
            best = child;
        }
    }
    best
}

// Selection : descente UCB1 jusqu'a un noeud terminal ou non pleinement expandu.
#[allow(dead_code)]
fn select(
    arena: &mut Vec<MctsNode>,
    mut index: usize,
    game: &mut Board,
    c: f64,
    stats: &mut Stats,
) -> usize {
    loop {
        let state = arena[index].state.clone();
        game.set_state(state);
        if game.is_terminal() {
            return index;
        }
        if arena[index].untried.is_none() {
            let state = arena[index].state.clone();
            game.set_state(state);
            arena[index].untried = Some(game.valid_moves());
        }
        let untried_empty = arena[index].untried.as_ref().map(|v| v.is_empty()).unwrap_or(false);
        let has_children = !arena[index].children.is_empty();
        if !(untried_empty && has_children) {
            return expand(arena, index, game, stats);
        }
        let best = best_child_ucb1(arena, index, c);
        index = best;
        let state = arena[index].state.clone();
        game.set_state(state);
    }
}

// Expansion : le dernier coup restant est pris (LIFO), sans tirage.
#[allow(dead_code)]
fn expand(
    arena: &mut Vec<MctsNode>,
    index: usize,
    game: &mut Board,
    stats: &mut Stats,
) -> usize {
    stats.nodes += 1;
    if arena[index].untried.is_none() {
        let state = arena[index].state.clone();
        game.set_state(state);
        arena[index].untried = Some(game.valid_moves());
    }
    let action = match arena[index].untried.as_mut() {
        Some(list) => list.pop(),
        None => None,
    };
    let action = match action {
        Some(a) => a,
        None => return index,
    };
    let state = arena[index].state.clone();
    game.set_state(state);
    game.drop_piece(action);
    let child_state = game.state();
    arena.push(MctsNode {
        state: child_state,
        action: Some(action),
        parent: Some(index),
        children: Vec::new(),
        visits: 0,
        wins: 0.0,
        untried: None,
    });
    let child = arena.len() - 1;
    arena[index].children.push(child);
    child
}

// Simulation : partie aleatoire jusqu'a un etat terminal.
#[allow(dead_code)]
fn simulate(game: &mut Board, rng: &mut Mt19937, stats: &mut Stats) -> f64 {
    stats.nodes += 1;
    let saved = game.state();
    let original_player = game.current_player;
    while !game.is_terminal() {
        let moves = game.valid_moves();
        if moves.is_empty() {
            break;
        }
        let mv = *rng.choice(&moves);
        game.drop_piece(mv);
    }
    let result = game.get_utility(original_player);
    game.set_state(saved);
    result
}

// Backpropagation : le resultat est inverse selon le joueur du noeud.
#[allow(dead_code)]
fn backpropagate(arena: &mut Vec<MctsNode>, mut index: usize, result: f64) {
    loop {
        arena[index].visits += 1;
        let player = arena[index].state.current_player;
        if player == -1 {
            arena[index].wins += result;
        } else {
            arena[index].wins -= result;
        }
        match arena[index].parent {
            Some(parent) => index = parent,
            None => break,
        }
    }
}

#[allow(dead_code)]
impl Mcts {
    // L'etat du jeu du caller n'est jamais modifie : il est sauvegarde en
    // entree et restaure en sortie.
    fn search(
        &mut self,
        game: &mut Board,
        iterations: u32,
        rng: &mut Mt19937,
    ) -> (Option<i32>, f64) {
        let c = self.c;
        let mut stats = Stats { nodes: 0 };
        let initial = game.state();
        let mut arena: Vec<MctsNode> = Vec::new();
        arena.push(MctsNode {
            state: initial.clone(),
            action: None,
            parent: None,
            children: Vec::new(),
            visits: 0,
            wins: 0.0,
            untried: None,
        });

        for _ in 0..iterations {
            let node = select(&mut arena, 0, game, c, &mut stats);
            let result = simulate(game, rng, &mut stats);
            backpropagate(&mut arena, node, result);
        }

        game.set_state(initial);
        self.stats = stats;

        if arena[0].children.is_empty() {
            return (None, 0.0);
        }
        let best = best_child_visits(&arena, 0);
        let visits = arena[best].visits;
        let win_rate = if visits > 0 {
            arena[best].wins / visits as f64
        } else {
            0.0
        };
        (arena[best].action, win_rate)
    }
}
// Separateur de milliers, comme le format `:,` de Python.
#[allow(dead_code)]
fn thousands(n: u64) -> String {
    let s = n.to_string();
    let len = s.len();
    let mut out = String::new();
    for (i, ch) in s.chars().enumerate() {
        if i > 0 && (len - i) % 3 == 0 {
            out.push(',');
        }
        out.push(ch);
    }
    out
}

// Liste d'entiers au format de la repr() d'une liste Python.
#[allow(dead_code)]
fn py_list_i32(values: &[i32]) -> String {
    let items: Vec<String> = values.iter().map(|v| v.to_string()).collect();
    format!("[{}]", items.join(", "))
}

// Un Option imprime comme un Optional Python.
#[allow(dead_code)]
fn py_opt_i32(value: Option<i32>) -> String {
    match value {
        Some(v) => v.to_string(),
        None => "None".to_string(),
    }
}
// Graine posee une seule fois : les cinq lignes du benchmark partagent le
// meme flux de tirage, comme dans le carnet Python.
let mut rng = Mt19937::new(42);

println!("Benchmark MCTS:");
{
    for iterations in [100u32, 500, 1000, 2000, 5000].iter() {
        // Etat initial identique pour chaque ligne : c'est le nombre
        // d'iterations qui varie, pas la position de depart.
        let mut game = Board::new();
        let mut mcts = Mcts { c: 1.41, stats: Stats { nodes: 0 } };

        let start = std::time::Instant::now();
        let (mv, win_rate) = mcts.search(&mut game, *iterations, &mut rng);
        let elapsed = start.elapsed().as_secs_f64();

        println!(
            "{} iterations: coup {}, taux {:.6}, {} noeuds, {:.4}s",
            iterations,
            py_opt_i32(mv),
            win_rate,
            mcts.stats.nodes,
            elapsed
        );
    }
};


Benchmark MCTS:


100 iterations: coup 0, taux 0.090909, 200 noeuds, 0.0009s


500 iterations: coup 4, taux 0.126984, 1000 noeuds, 0.0065s


1000 iterations: coup 6, taux 0.161148, 2000 noeuds, 0.0117s


2000 iterations: coup 5, taux 0.024450, 4000 noeuds, 0.0213s


5000 iterations: coup 0, taux 0.015695, 10000 noeuds, 0.0466s


### Interprétation : Benchmark MCTS

**Sortie obtenue** : performance de MCTS sur la position initiale (plateau vide) avec différents nombres d'itérations (graine posee au début du benchmark MCTS : table reproductible).

**Lecture du tableau** :
- **Coup** : colonne choisie par MCTS comme meilleur premier coup pour le joueur 1. La théorie (Allis 1988) donne la colonne 3 (centre) comme optimale ; sur la sortie, les colonnes choisies restent erratiques sur toute la grille de budgets (0, 4, 6, 5, 0) -- voir « Points clés ».
- **Taux victoire** : ratio `wins/visites` au noeud du coup choisi a la racine. Deux conventions a lire avant d'interpreter la colonne : (1) la backpropagation alternée (`_backpropagate`) stocke au noeud enfant la valeur **du point de vue du joueur qui y joue** -- l'adversaire du joueur 1 -- et (2) cette valeur est **signee** (+1 victoire, 0 nul, -1 defaite). La colonne mesure donc l'estimation esperee de l'adversaire après le coup choisi : les valeurs observees (+0.016 a +0.161) restent proches de zero -- l'adversaire n'a statistiquement ni avantage net ni desavantage net sous rollouts uniformes. Ce n'est PAS une probabilite de victoire du joueur 1.
- **Noeuds explores** : compteur cumule expansions + simulations. Pour `itérations=N`, on a typiquement ~2N (une expansion + une simulation par tour) -- exactement 2N sur chaque ligne de la sortie.
- **Temps** : temps mur. Croît ~linéairement avec le nombre d'itérations ; la valeur exacte dépend du poste et de la charge, elle se lit dans la sortie ci-dessous.

**Ce que montre la sortie** :
1. **Pas de convergence vers le centre** : le coup choisi varie d'un budget a l'autre (0, 4, 6, 5, 0). Avec des rollouts uniformement aléatoires, le signal est trop pauvre pour discriminer les colonnes de Connect Four ; a ces budgets, MCTS se comporte proche d'un choix quasi aléatoire -- le tournoi de la section suivante le confirme (MCTS(1000) finit dernier).
2. **Taux victoire faible, sans tendance** : +0.016 a +0.161 sur toute la grille, sans affinement net quand le budget monte -- coherent avec un signal de rollout pauvre, pas avec une estimation qui convergerait.
3. **Noeuds explores** : exactement 2 x itérations sur chaque ligne (200, 1000, 2000, 4000, 10000) -- l'invariant structurel du benchmark.

**Points clés** :
- **cohérence garantie** : grâce à la sauvegarde et à la restauration de l'état dans `Mcts::search` et à la création d'un `Board::new()` neuf par ligne du benchmark, chaque ligne du tableau correspond bien a un benchmark sur la **même position de départ** (plateau vide). Sans cette restauration, l'état du jeu derivait entre itérations vers des positions arbitraires, faussant la comparaison.
- **MCTS est stochastique** : la graine posee au début du benchmark MCTS rend la table reproductible d'une exécution a l'autre ; sans graine, coups et taux fluctuent a chaque re-exécution (rollouts aléatoires).
- **Connect Four est résolu** : le joueur 1 gagne avec jeu parfait en commencant par la colonne 3 (Allis 1988). MCTS sans heuristique d'évaluation ne le découvre qu'avec beaucoup de rollouts -- la sortie ci-dessus en est l'illustration directe ; pour un agent réaliste sur Connect Four, on couplerait MCTS a un réseau de neurones (cf AlphaZero) ou on utiliserait Alpha-Beta avec table de transposition.

> **Note technique** : les rollouts sont **uniformêment aléatoires** ici (cf la simulation de `Mcts::search`). Cela donne un signal pauvre sur Connect Four ou la victoire dépend de patterns précis (4-en-ligne). Une amélioration classique consiste a biaiser les rollouts vers des coups gagnants/bloquants immédiats (heuristique de simulation), ou a remplacer la simulation par l'évaluation d'un réseau de neurones entraine.

Comparaison des résultats : les figures du carnet Python sont remplacées par leurs données imprimées.

In [10]:
// Moteur de jeu Connect Four : grille de 6 lignes x 7 colonnes.
// 0 = case vide, 1 = joueur MAX (X), -1 = joueur MIN (O).
#[allow(dead_code)]
#[derive(Clone)]
struct Board {
    grid: [[i32; 7]; 6],
    current_player: i32,
}

#[allow(dead_code)]
impl Board {
    fn new() -> Board {
        Board { grid: [[0; 7]; 6], current_player: 1 }
    }
    fn state(&self) -> Board {
        self.clone()
    }
    fn set_state(&mut self, s: Board) {
        self.grid = s.grid;
        self.current_player = s.current_player;
    }
    // Colonnes dont la case du haut est libre.
    fn valid_moves(&self) -> Vec<i32> {
        let mut moves = Vec::new();
        for c in 0..7 {
            if self.grid[0][c] == 0 {
                moves.push(c as i32);
            }
        }
        moves
    }
    // Pose le jeton par gravite ; le joueur courant change APRES la pose.
    fn drop_piece(&mut self, col: i32) -> bool {
        if col < 0 || col >= 7 {
            return false;
        }
        let c = col as usize;
        if self.grid[0][c] != 0 {
            return false;
        }
        for row in (0..6).rev() {
            if self.grid[row][c] == 0 {
                self.grid[row][c] = self.current_player;
                self.current_player *= -1;
                return true;
            }
        }
        false
    }
    // Retire le jeton du haut de la colonne ; le joueur courant change aussi.
    fn undo_move(&mut self, col: i32) {
        let c = col as usize;
        for row in 0..6 {
            if self.grid[row][c] != 0 {
                self.grid[row][c] = 0;
                self.current_player *= -1;
                return;
            }
        }
    }
    // Ordre des directions : horizontales, verticales, diagonales bas-droite,
    // diagonales bas-gauche.
    fn check_winner(&self) -> i32 {
        for r in 0..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r][c + 1], self.grid[r][c + 2], self.grid[r][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..7 {
                let w = [self.grid[r][c], self.grid[r + 1][c], self.grid[r + 2][c], self.grid[r + 3][c]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r + 1][c + 1], self.grid[r + 2][c + 2], self.grid[r + 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 3..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r - 1][c + 1], self.grid[r - 2][c + 2], self.grid[r - 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        0
    }
    fn is_terminal(&self) -> bool {
        self.check_winner() != 0 || self.valid_moves().is_empty()
    }
    fn get_utility(&self, maximizing_player: i32) -> f64 {
        (self.check_winner() * maximizing_player) as f64
    }
    fn display(&self) -> String {
        let sym = |c: i32| -> char {
            if c == 0 {
                '.'
            } else if c == 1 {
                'X'
            } else {
                'O'
            }
        };
        let mut lines: Vec<String> = Vec::new();
        for r in 0..6 {
            let cells: Vec<String> = (0..7).map(|c| sym(self.grid[r][c]).to_string()).collect();
            lines.push(cells.join(" "));
        }
        let cols: Vec<String> = (0..7).map(|c| c.to_string()).collect();
        lines.push(cols.join(" "));
        lines.join("\n")
    }
}
// Heuristique : score des fenetres de 4 cases + controle du centre.
#[allow(dead_code)]
fn evaluate_window(window: &[i32; 4], player: i32) -> i64 {
    let mut score: i64 = 0;
    let opponent = -player;

    let player_count = window.iter().filter(|&&x| x == player).count();
    let opponent_count = window.iter().filter(|&&x| x == opponent).count();
    let empty_count = window.iter().filter(|&&x| x == 0).count();

    // Fenetre gagnante
    if player_count == 4 {
        score += 100;
    } else if player_count == 3 && empty_count == 1 {
        score += 5;
    } else if player_count == 2 && empty_count == 2 {
        score += 2;
    }

    // Bloquer l'adversaire (3 jetons adverses + 1 vide)
    if opponent_count == 3 && empty_count == 1 {
        score -= 4;
    }

    score
}

#[allow(dead_code)]
fn evaluate_position(game: &Board, player: i32) -> f64 {
    if game.is_terminal() {
        return game.get_utility(player) * 1000.0;
    }

    let mut score: i64 = 0;

    // Bonus pour le centre (colonne 3)
    for r in 0..6 {
        if game.grid[r][3] == player {
            score += 3;
        }
    }

    // Horizontales
    for r in 0..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r][c + 1], game.grid[r][c + 2], game.grid[r][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Verticales
    for r in 0..3 {
        for c in 0..7 {
            let w = [game.grid[r][c], game.grid[r + 1][c], game.grid[r + 2][c], game.grid[r + 3][c]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-droite
    for r in 0..3 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r + 1][c + 1], game.grid[r + 2][c + 2], game.grid[r + 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-gauche
    for r in 3..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r - 1][c + 1], game.grid[r - 2][c + 2], game.grid[r - 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    score as f64
}
// Statistiques de recherche et Minimax (sans tri des coups).
#[allow(dead_code)]
#[derive(Default)]
struct Stats {
    nodes: u64,
}

// Minimax : les coups sont parcourus dans l'ordre brut des colonnes,
// et le premier maximum strict est retenu.
#[allow(dead_code)]
fn minimax(
    game: &mut Board,
    depth: i32,
    maximizing: bool,
    stats: &mut Stats,
    player: i32,
) -> (f64, Option<i32>) {
    stats.nodes += 1;

    if depth == 0 || game.is_terminal() {
        if game.is_terminal() {
            return (game.get_utility(player) * 1000.0, None);
        }
        return (evaluate_position(game, player), None);
    }

    let valid_moves = game.valid_moves();

    if maximizing {
        let mut best_value = f64::NEG_INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, false, stats, player);
            game.undo_move(mv);
            if value > best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    } else {
        let mut best_value = f64::INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, true, stats, player);
            game.undo_move(mv);
            if value < best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    }
}
// Alpha-Beta : les coups sont ordonnes par proximite au centre (tri stable),
// et la coupe s'applique quand beta <= alpha.
#[allow(dead_code)]
fn alpha_beta(
    game: &mut Board,
    depth: i32,
    mut alpha: f64,
    mut beta: f64,
    maximizing: bool,
    stats: &mut Stats,
    player: i32,
) -> (f64, Option<i32>) {
    stats.nodes += 1;

    if depth == 0 || game.is_terminal() {
        if game.is_terminal() {
            return (game.get_utility(player) * 1000.0, None);
        }
        return (evaluate_position(game, player), None);
    }

    let mut valid_moves = game.valid_moves();
    valid_moves.sort_by_key(|&c| (c - 3).abs());

    if maximizing {
        let mut best_value = f64::NEG_INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = alpha_beta(game, depth - 1, alpha, beta, false, stats, player);
            game.undo_move(mv);
            if value > best_value {
                best_value = value;
                best_move = Some(mv);
            }
            if best_value > alpha {
                alpha = best_value;
            }
            if beta <= alpha {
                break;
            }
        }
        (best_value, best_move)
    } else {
        let mut best_value = f64::INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = alpha_beta(game, depth - 1, alpha, beta, true, stats, player);
            game.undo_move(mv);
            if value < best_value {
                best_value = value;
                best_move = Some(mv);
            }
            if best_value < beta {
                beta = best_value;
            }
            if beta <= alpha {
                break;
            }
        }
        (best_value, best_move)
    }
}
// Les deux figures du carnet Python sont remplacees par leurs donnees
// imprimees : meme information, aucun fichier ecrit.
let mut game = Board::new();
let mut mm_nodes: Vec<u64> = Vec::new();
let mut ab_nodes: Vec<u64> = Vec::new();
let mut mm_times: Vec<f64> = Vec::new();
let mut ab_times: Vec<f64> = Vec::new();
let mut speedups: Vec<f64> = Vec::new();

for depth in 1..=6 {
    let mut stats_mm = Stats { nodes: 0 };
    let start_mm = std::time::Instant::now();
    let _ = minimax(&mut game, depth, true, &mut stats_mm, 1);
    mm_times.push(start_mm.elapsed().as_secs_f64());

    let mut stats_ab = Stats { nodes: 0 };
    let start_ab = std::time::Instant::now();
    let _ = alpha_beta(
        &mut game,
        depth,
        f64::NEG_INFINITY,
        f64::INFINITY,
        true,
        &mut stats_ab,
        1,
    );
    ab_times.push(start_ab.elapsed().as_secs_f64());

    let ratio = stats_mm.nodes as f64 / stats_ab.nodes.max(1) as f64;
    mm_nodes.push(stats_mm.nodes);
    ab_nodes.push(stats_ab.nodes);
    speedups.push(ratio);
}

// Fonctions de formatage : evcxr ne sait pas persister une fermeture, les
// helpers sont donc des fonctions d'items.
#[allow(dead_code)]
fn join_counts(values: &Vec<u64>) -> String {
    values
        .iter()
        .map(|n| n.to_string())
        .collect::<Vec<String>>()
        .join(", ")
}

#[allow(dead_code)]
fn join_times(values: &Vec<f64>) -> String {
    values
        .iter()
        .map(|t| format!("{:.4}", t))
        .collect::<Vec<String>>()
        .join(", ")
}

#[allow(dead_code)]
fn join_speedups(values: &Vec<f64>) -> String {
    values
        .iter()
        .map(|r| {
            if r.fract() == 0.0 {
                format!("{:.1}", r)
            } else {
                format!("{:.2}", r)
            }
        })
        .collect::<Vec<String>>()
        .join(", ")
}

println!(
    "Noeuds par profondeur : Minimax [{}], Alpha-Beta [{}]",
    join_counts(&mm_nodes),
    join_counts(&ab_nodes)
);
println!(
    "Temps par profondeur (s) : Minimax [{}], Alpha-Beta [{}]",
    join_times(&mm_times),
    join_times(&ab_times)
);
println!("Speedup noeuds : [{}]", join_speedups(&speedups));


Noeuds par profondeur : Minimax [8, 57, 400, 2801, 19608, 137257], Alpha-Beta [8, 21, 82, 178, 641, 1370]


Temps par profondeur (s) : Minimax [0.0000, 0.0000, 0.0002, 0.0015, 0.0119, 0.0949], Alpha-Beta [0.0000, 0.0000, 0.0000, 0.0001, 0.0005, 0.0006]


Speedup noeuds : [1.0, 2.71, 4.88, 15.74, 30.59, 100.19]


### Interprétation : Données de la comparaison

**Sortie obtenue** : les deux figures du carnet Python — nœuds explorés par profondeur et temps d'exécution par profondeur — sont **remplacées par leurs données imprimées** : ce carnet n'émet aucune image et n'écrit aucun fichier.

| Aspect | observation | Signification |
|--------|-------------|---------------|
| **Noeuds explores** | croissance exponentielle pour Minimax, bien plus lente pour Alpha-Beta (invariants structurels) | Elagage de plus en plus efficace avec la profondeur |
| **Temps d'exécution** | ecart runtime net entre les petites et les grandes profondeurs (runtime *machine-dep*) | Le runtime depend du poste et de la charge |
| **Speedup noeuds** | 1x (d=1) a 100x (d=6) -- invariant structurel déterministe | L'elagage augmente avec la profondeur |
| **echelle log** | nécessaire pour visualiser la différence | différence de 2 ordres de grandeur |

**Points clés** :
1. **Explosion combinatoire** : Minimax devient impraticable des les plus grandes profondeurs (runtime *machine-dep* sur ce poste)
2. **Efficacite Alpha-Beta** : 100x moins de noeuds explores (invariant structurel), speedup wall-clock *machine-dep* aux grandes profondeurs
3. **Seuil pratique** : Pour Connect Four, Alpha-Beta sur 6 a 8 plis est realiste, Minimax au-dela de 5 ne l'est pas
4. **Ordre de grandeur** : L'echelle logarithmique masque la difference reelle (137k vs 1.4k noeuds, invariant structurel)

> **Note technique** : Le **speedup structurel** (rapport des noeuds explores, 1x a 100x entre la profondeur minimale et la profondeur maximale) augmente avec la profondeur car l'elagage a plus d'opportunites de couper des branches. A la profondeur minimale, aucun elagage n'est possible (toutes les feuilles sont au même niveau). L'ordonnancement des coups (centre en priorite) maximise cet elagage. Le **speedup wall-clock** (rapport des temps d'exécution, runtime *machine-dep* sur ce poste) suit le même pattern mais avec une amplitude qui depend du runtime, de la charge système et de la taille de l'instance ; il **ne survit pas** a une re-exécution sur une autre machine, même si l'ordre de grandeur est preserve. Pour observer vos propres timings, executez la cellule de benchmark ci-dessus.

## 7. Tournoi entre algorithmes
Faisons jouer les algorithmes les uns contre les autrès pour mesurer leur force relative.

In [11]:
// Moteur de jeu Connect Four : grille de 6 lignes x 7 colonnes.
// 0 = case vide, 1 = joueur MAX (X), -1 = joueur MIN (O).
#[allow(dead_code)]
#[derive(Clone)]
struct Board {
    grid: [[i32; 7]; 6],
    current_player: i32,
}

#[allow(dead_code)]
impl Board {
    fn new() -> Board {
        Board { grid: [[0; 7]; 6], current_player: 1 }
    }
    fn state(&self) -> Board {
        self.clone()
    }
    fn set_state(&mut self, s: Board) {
        self.grid = s.grid;
        self.current_player = s.current_player;
    }
    // Colonnes dont la case du haut est libre.
    fn valid_moves(&self) -> Vec<i32> {
        let mut moves = Vec::new();
        for c in 0..7 {
            if self.grid[0][c] == 0 {
                moves.push(c as i32);
            }
        }
        moves
    }
    // Pose le jeton par gravite ; le joueur courant change APRES la pose.
    fn drop_piece(&mut self, col: i32) -> bool {
        if col < 0 || col >= 7 {
            return false;
        }
        let c = col as usize;
        if self.grid[0][c] != 0 {
            return false;
        }
        for row in (0..6).rev() {
            if self.grid[row][c] == 0 {
                self.grid[row][c] = self.current_player;
                self.current_player *= -1;
                return true;
            }
        }
        false
    }
    // Retire le jeton du haut de la colonne ; le joueur courant change aussi.
    fn undo_move(&mut self, col: i32) {
        let c = col as usize;
        for row in 0..6 {
            if self.grid[row][c] != 0 {
                self.grid[row][c] = 0;
                self.current_player *= -1;
                return;
            }
        }
    }
    // Ordre des directions : horizontales, verticales, diagonales bas-droite,
    // diagonales bas-gauche.
    fn check_winner(&self) -> i32 {
        for r in 0..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r][c + 1], self.grid[r][c + 2], self.grid[r][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..7 {
                let w = [self.grid[r][c], self.grid[r + 1][c], self.grid[r + 2][c], self.grid[r + 3][c]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r + 1][c + 1], self.grid[r + 2][c + 2], self.grid[r + 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 3..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r - 1][c + 1], self.grid[r - 2][c + 2], self.grid[r - 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        0
    }
    fn is_terminal(&self) -> bool {
        self.check_winner() != 0 || self.valid_moves().is_empty()
    }
    fn get_utility(&self, maximizing_player: i32) -> f64 {
        (self.check_winner() * maximizing_player) as f64
    }
    fn display(&self) -> String {
        let sym = |c: i32| -> char {
            if c == 0 {
                '.'
            } else if c == 1 {
                'X'
            } else {
                'O'
            }
        };
        let mut lines: Vec<String> = Vec::new();
        for r in 0..6 {
            let cells: Vec<String> = (0..7).map(|c| sym(self.grid[r][c]).to_string()).collect();
            lines.push(cells.join(" "));
        }
        let cols: Vec<String> = (0..7).map(|c| c.to_string()).collect();
        lines.push(cols.join(" "));
        lines.join("\n")
    }
}
// Heuristique : score des fenetres de 4 cases + controle du centre.
#[allow(dead_code)]
fn evaluate_window(window: &[i32; 4], player: i32) -> i64 {
    let mut score: i64 = 0;
    let opponent = -player;

    let player_count = window.iter().filter(|&&x| x == player).count();
    let opponent_count = window.iter().filter(|&&x| x == opponent).count();
    let empty_count = window.iter().filter(|&&x| x == 0).count();

    // Fenetre gagnante
    if player_count == 4 {
        score += 100;
    } else if player_count == 3 && empty_count == 1 {
        score += 5;
    } else if player_count == 2 && empty_count == 2 {
        score += 2;
    }

    // Bloquer l'adversaire (3 jetons adverses + 1 vide)
    if opponent_count == 3 && empty_count == 1 {
        score -= 4;
    }

    score
}

#[allow(dead_code)]
fn evaluate_position(game: &Board, player: i32) -> f64 {
    if game.is_terminal() {
        return game.get_utility(player) * 1000.0;
    }

    let mut score: i64 = 0;

    // Bonus pour le centre (colonne 3)
    for r in 0..6 {
        if game.grid[r][3] == player {
            score += 3;
        }
    }

    // Horizontales
    for r in 0..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r][c + 1], game.grid[r][c + 2], game.grid[r][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Verticales
    for r in 0..3 {
        for c in 0..7 {
            let w = [game.grid[r][c], game.grid[r + 1][c], game.grid[r + 2][c], game.grid[r + 3][c]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-droite
    for r in 0..3 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r + 1][c + 1], game.grid[r + 2][c + 2], game.grid[r + 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-gauche
    for r in 3..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r - 1][c + 1], game.grid[r - 2][c + 2], game.grid[r - 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    score as f64
}
// Statistiques de recherche et Minimax (sans tri des coups).
#[allow(dead_code)]
#[derive(Default)]
struct Stats {
    nodes: u64,
}

// Minimax : les coups sont parcourus dans l'ordre brut des colonnes,
// et le premier maximum strict est retenu.
#[allow(dead_code)]
fn minimax(
    game: &mut Board,
    depth: i32,
    maximizing: bool,
    stats: &mut Stats,
    player: i32,
) -> (f64, Option<i32>) {
    stats.nodes += 1;

    if depth == 0 || game.is_terminal() {
        if game.is_terminal() {
            return (game.get_utility(player) * 1000.0, None);
        }
        return (evaluate_position(game, player), None);
    }

    let valid_moves = game.valid_moves();

    if maximizing {
        let mut best_value = f64::NEG_INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, false, stats, player);
            game.undo_move(mv);
            if value > best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    } else {
        let mut best_value = f64::INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, true, stats, player);
            game.undo_move(mv);
            if value < best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    }
}
// Alpha-Beta : les coups sont ordonnes par proximite au centre (tri stable),
// et la coupe s'applique quand beta <= alpha.
#[allow(dead_code)]
fn alpha_beta(
    game: &mut Board,
    depth: i32,
    mut alpha: f64,
    mut beta: f64,
    maximizing: bool,
    stats: &mut Stats,
    player: i32,
) -> (f64, Option<i32>) {
    stats.nodes += 1;

    if depth == 0 || game.is_terminal() {
        if game.is_terminal() {
            return (game.get_utility(player) * 1000.0, None);
        }
        return (evaluate_position(game, player), None);
    }

    let mut valid_moves = game.valid_moves();
    valid_moves.sort_by_key(|&c| (c - 3).abs());

    if maximizing {
        let mut best_value = f64::NEG_INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = alpha_beta(game, depth - 1, alpha, beta, false, stats, player);
            game.undo_move(mv);
            if value > best_value {
                best_value = value;
                best_move = Some(mv);
            }
            if best_value > alpha {
                alpha = best_value;
            }
            if beta <= alpha {
                break;
            }
        }
        (best_value, best_move)
    } else {
        let mut best_value = f64::INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = alpha_beta(game, depth - 1, alpha, beta, true, stats, player);
            game.undo_move(mv);
            if value < best_value {
                best_value = value;
                best_move = Some(mv);
            }
            if best_value < beta {
                beta = best_value;
            }
            if beta <= alpha {
                break;
            }
        }
        (best_value, best_move)
    }
}
// MT19937 compatible avec random.Random de CPython (init_by_array).
// Le carnet sème 42 avant chaque demonstration stochastique : c'est ce
// generateur qui rend les sorties reproductibles.
#[allow(dead_code)]
struct Mt19937 {
    mt: [u32; 624],
    idx: usize,
}

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32
                .wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30))
                .wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 {
        Mt19937::new_by_array(&[seed])
    }
    fn new_by_array(key: &[u32]) -> Mt19937 {
        let mut mt = Mt19937::init_genrand(19650218);
        let mut i: usize = 1;
        let mut j: usize = 0;
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j])
                .wrapping_add(j as u32);
            i += 1;
            j += 1;
            if i >= 624 {
                mt[0] = mt[623];
                i = 1;
            }
            if j >= key.len() {
                j = 0;
            }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 {
                mt[0] = mt[623];
                i = 1;
            }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 {
                    v ^= 0x9908_b0df;
                }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn getrandbits(&mut self, k: u32) -> u64 {
        if k <= 32 {
            return (self.next_u32() >> (32 - k)) as u64;
        }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    // Rejet, comme _randbelow de CPython.
    fn randbelow(&mut self, n: u64) -> u64 {
        if n == 0 {
            return 0;
        }
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n {
            r = self.getrandbits(k);
        }
        r
    }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T {
        &seq[self.randbelow(seq.len() as u64) as usize]
    }
}
// Monte Carlo Tree Search. L'arbre est un arena : chaque noeud porte son
// parent, ses enfants (dans l'ordre d'insertion) et ses compteurs.
#[allow(dead_code)]
#[derive(Clone)]
struct MctsNode {
    state: Board,
    action: Option<i32>,
    parent: Option<usize>,
    children: Vec<usize>,
    visits: u64,
    wins: f64,
    untried: Option<Vec<i32>>,
}

#[allow(dead_code)]
struct Mcts {
    c: f64,
    stats: Stats,
}

// Score UCB1 : infini tant que le noeud n'a pas ete visite.
#[allow(dead_code)]
fn ucb1(arena: &[MctsNode], index: usize, c: f64) -> f64 {
    let node = &arena[index];
    if node.visits == 0 {
        return f64::INFINITY;
    }
    let exploitation = node.wins / node.visits as f64;
    let parent_visits = match node.parent {
        Some(p) => arena[p].visits,
        None => 0,
    };
    if parent_visits == 0 {
        return f64::INFINITY;
    }
    let exploration = c * ((parent_visits as f64).ln() / node.visits as f64).sqrt();
    exploitation + exploration
}

// Premier maximum strict (comme max() de Python sur un dict ordonne).
#[allow(dead_code)]
fn best_child_ucb1(arena: &[MctsNode], index: usize, c: f64) -> usize {
    let mut best = arena[index].children[0];
    let mut best_score = ucb1(arena, best, c);
    for &child in arena[index].children.iter().skip(1) {
        let score = ucb1(arena, child, c);
        if score > best_score {
            best_score = score;
            best = child;
        }
    }
    best
}

// Premier maximum strict du nombre de visites (coup final).
#[allow(dead_code)]
fn best_child_visits(arena: &[MctsNode], index: usize) -> usize {
    let mut best = arena[index].children[0];
    let mut best_visits = arena[best].visits;
    for &child in arena[index].children.iter().skip(1) {
        if arena[child].visits > best_visits {
            best_visits = arena[child].visits;
            best = child;
        }
    }
    best
}

// Selection : descente UCB1 jusqu'a un noeud terminal ou non pleinement expandu.
#[allow(dead_code)]
fn select(
    arena: &mut Vec<MctsNode>,
    mut index: usize,
    game: &mut Board,
    c: f64,
    stats: &mut Stats,
) -> usize {
    loop {
        let state = arena[index].state.clone();
        game.set_state(state);
        if game.is_terminal() {
            return index;
        }
        if arena[index].untried.is_none() {
            let state = arena[index].state.clone();
            game.set_state(state);
            arena[index].untried = Some(game.valid_moves());
        }
        let untried_empty = arena[index].untried.as_ref().map(|v| v.is_empty()).unwrap_or(false);
        let has_children = !arena[index].children.is_empty();
        if !(untried_empty && has_children) {
            return expand(arena, index, game, stats);
        }
        let best = best_child_ucb1(arena, index, c);
        index = best;
        let state = arena[index].state.clone();
        game.set_state(state);
    }
}

// Expansion : le dernier coup restant est pris (LIFO), sans tirage.
#[allow(dead_code)]
fn expand(
    arena: &mut Vec<MctsNode>,
    index: usize,
    game: &mut Board,
    stats: &mut Stats,
) -> usize {
    stats.nodes += 1;
    if arena[index].untried.is_none() {
        let state = arena[index].state.clone();
        game.set_state(state);
        arena[index].untried = Some(game.valid_moves());
    }
    let action = match arena[index].untried.as_mut() {
        Some(list) => list.pop(),
        None => None,
    };
    let action = match action {
        Some(a) => a,
        None => return index,
    };
    let state = arena[index].state.clone();
    game.set_state(state);
    game.drop_piece(action);
    let child_state = game.state();
    arena.push(MctsNode {
        state: child_state,
        action: Some(action),
        parent: Some(index),
        children: Vec::new(),
        visits: 0,
        wins: 0.0,
        untried: None,
    });
    let child = arena.len() - 1;
    arena[index].children.push(child);
    child
}

// Simulation : partie aleatoire jusqu'a un etat terminal.
#[allow(dead_code)]
fn simulate(game: &mut Board, rng: &mut Mt19937, stats: &mut Stats) -> f64 {
    stats.nodes += 1;
    let saved = game.state();
    let original_player = game.current_player;
    while !game.is_terminal() {
        let moves = game.valid_moves();
        if moves.is_empty() {
            break;
        }
        let mv = *rng.choice(&moves);
        game.drop_piece(mv);
    }
    let result = game.get_utility(original_player);
    game.set_state(saved);
    result
}

// Backpropagation : le resultat est inverse selon le joueur du noeud.
#[allow(dead_code)]
fn backpropagate(arena: &mut Vec<MctsNode>, mut index: usize, result: f64) {
    loop {
        arena[index].visits += 1;
        let player = arena[index].state.current_player;
        if player == -1 {
            arena[index].wins += result;
        } else {
            arena[index].wins -= result;
        }
        match arena[index].parent {
            Some(parent) => index = parent,
            None => break,
        }
    }
}

#[allow(dead_code)]
impl Mcts {
    // L'etat du jeu du caller n'est jamais modifie : il est sauvegarde en
    // entree et restaure en sortie.
    fn search(
        &mut self,
        game: &mut Board,
        iterations: u32,
        rng: &mut Mt19937,
    ) -> (Option<i32>, f64) {
        let c = self.c;
        let mut stats = Stats { nodes: 0 };
        let initial = game.state();
        let mut arena: Vec<MctsNode> = Vec::new();
        arena.push(MctsNode {
            state: initial.clone(),
            action: None,
            parent: None,
            children: Vec::new(),
            visits: 0,
            wins: 0.0,
            untried: None,
        });

        for _ in 0..iterations {
            let node = select(&mut arena, 0, game, c, &mut stats);
            let result = simulate(game, rng, &mut stats);
            backpropagate(&mut arena, node, result);
        }

        game.set_state(initial);
        self.stats = stats;

        if arena[0].children.is_empty() {
            return (None, 0.0);
        }
        let best = best_child_visits(&arena, 0);
        let visits = arena[best].visits;
        let win_rate = if visits > 0 {
            arena[best].wins / visits as f64
        } else {
            0.0
        };
        (arena[best].action, win_rate)
    }
}
// Separateur de milliers, comme le format `:,` de Python.
#[allow(dead_code)]
fn thousands(n: u64) -> String {
    let s = n.to_string();
    let len = s.len();
    let mut out = String::new();
    for (i, ch) in s.chars().enumerate() {
        if i > 0 && (len - i) % 3 == 0 {
            out.push(',');
        }
        out.push(ch);
    }
    out
}

// Liste d'entiers au format de la repr() d'une liste Python.
#[allow(dead_code)]
fn py_list_i32(values: &[i32]) -> String {
    let items: Vec<String> = values.iter().map(|v| v.to_string()).collect();
    format!("[{}]", items.join(", "))
}

// Un Option imprime comme un Optional Python.
#[allow(dead_code)]
fn py_opt_i32(value: Option<i32>) -> String {
    match value {
        Some(v) => v.to_string(),
        None => "None".to_string(),
    }
}
#[allow(dead_code)]
fn agent_random(game: &Board, rng: &mut Mt19937) -> Option<i32> {
    let moves = game.valid_moves();
    if moves.is_empty() {
        None
    } else {
        Some(*rng.choice(&moves))
    }
}

#[allow(dead_code)]
fn agent_minimax_depth4(game: &mut Board) -> Option<i32> {
    let maximizing = game.current_player == 1;
    let mut stats = Stats { nodes: 0 };
    let (_, mv) = minimax(game, 4, maximizing, &mut stats, 1);
    mv
}

#[allow(dead_code)]
fn agent_alphabeta_depth4(game: &mut Board) -> Option<i32> {
    let maximizing = game.current_player == 1;
    let mut stats = Stats { nodes: 0 };
    let (_, mv) = alpha_beta(
        game,
        4,
        f64::NEG_INFINITY,
        f64::INFINITY,
        maximizing,
        &mut stats,
        1,
    );
    mv
}

#[allow(dead_code)]
fn agent_mcts_1000(game: &mut Board, rng: &mut Mt19937) -> Option<i32> {
    let mut mcts = Mcts { c: 1.41, stats: Stats { nodes: 0 } };
    let (mv, _) = mcts.search(game, 1000, rng);
    mv
}

println!("Agents definis: Random, Minimax(d=4), AlphaBeta(d=4), MCTS(1000)");


Agents definis: Random, Minimax(d=4), AlphaBeta(d=4), MCTS(1000)


Simulation d'un tournoi complet entre agents.

In [12]:
// Moteur de jeu Connect Four : grille de 6 lignes x 7 colonnes.
// 0 = case vide, 1 = joueur MAX (X), -1 = joueur MIN (O).
#[allow(dead_code)]
#[derive(Clone)]
struct Board {
    grid: [[i32; 7]; 6],
    current_player: i32,
}

#[allow(dead_code)]
impl Board {
    fn new() -> Board {
        Board { grid: [[0; 7]; 6], current_player: 1 }
    }
    fn state(&self) -> Board {
        self.clone()
    }
    fn set_state(&mut self, s: Board) {
        self.grid = s.grid;
        self.current_player = s.current_player;
    }
    // Colonnes dont la case du haut est libre.
    fn valid_moves(&self) -> Vec<i32> {
        let mut moves = Vec::new();
        for c in 0..7 {
            if self.grid[0][c] == 0 {
                moves.push(c as i32);
            }
        }
        moves
    }
    // Pose le jeton par gravite ; le joueur courant change APRES la pose.
    fn drop_piece(&mut self, col: i32) -> bool {
        if col < 0 || col >= 7 {
            return false;
        }
        let c = col as usize;
        if self.grid[0][c] != 0 {
            return false;
        }
        for row in (0..6).rev() {
            if self.grid[row][c] == 0 {
                self.grid[row][c] = self.current_player;
                self.current_player *= -1;
                return true;
            }
        }
        false
    }
    // Retire le jeton du haut de la colonne ; le joueur courant change aussi.
    fn undo_move(&mut self, col: i32) {
        let c = col as usize;
        for row in 0..6 {
            if self.grid[row][c] != 0 {
                self.grid[row][c] = 0;
                self.current_player *= -1;
                return;
            }
        }
    }
    // Ordre des directions : horizontales, verticales, diagonales bas-droite,
    // diagonales bas-gauche.
    fn check_winner(&self) -> i32 {
        for r in 0..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r][c + 1], self.grid[r][c + 2], self.grid[r][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..7 {
                let w = [self.grid[r][c], self.grid[r + 1][c], self.grid[r + 2][c], self.grid[r + 3][c]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 0..3 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r + 1][c + 1], self.grid[r + 2][c + 2], self.grid[r + 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        for r in 3..6 {
            for c in 0..4 {
                let w = [self.grid[r][c], self.grid[r - 1][c + 1], self.grid[r - 2][c + 2], self.grid[r - 3][c + 3]];
                if w.iter().copied().sum::<i32>().abs() == 4 && w.iter().all(|&x| x == w[0]) {
                    return w[0];
                }
            }
        }
        0
    }
    fn is_terminal(&self) -> bool {
        self.check_winner() != 0 || self.valid_moves().is_empty()
    }
    fn get_utility(&self, maximizing_player: i32) -> f64 {
        (self.check_winner() * maximizing_player) as f64
    }
    fn display(&self) -> String {
        let sym = |c: i32| -> char {
            if c == 0 {
                '.'
            } else if c == 1 {
                'X'
            } else {
                'O'
            }
        };
        let mut lines: Vec<String> = Vec::new();
        for r in 0..6 {
            let cells: Vec<String> = (0..7).map(|c| sym(self.grid[r][c]).to_string()).collect();
            lines.push(cells.join(" "));
        }
        let cols: Vec<String> = (0..7).map(|c| c.to_string()).collect();
        lines.push(cols.join(" "));
        lines.join("\n")
    }
}
// Heuristique : score des fenetres de 4 cases + controle du centre.
#[allow(dead_code)]
fn evaluate_window(window: &[i32; 4], player: i32) -> i64 {
    let mut score: i64 = 0;
    let opponent = -player;

    let player_count = window.iter().filter(|&&x| x == player).count();
    let opponent_count = window.iter().filter(|&&x| x == opponent).count();
    let empty_count = window.iter().filter(|&&x| x == 0).count();

    // Fenetre gagnante
    if player_count == 4 {
        score += 100;
    } else if player_count == 3 && empty_count == 1 {
        score += 5;
    } else if player_count == 2 && empty_count == 2 {
        score += 2;
    }

    // Bloquer l'adversaire (3 jetons adverses + 1 vide)
    if opponent_count == 3 && empty_count == 1 {
        score -= 4;
    }

    score
}

#[allow(dead_code)]
fn evaluate_position(game: &Board, player: i32) -> f64 {
    if game.is_terminal() {
        return game.get_utility(player) * 1000.0;
    }

    let mut score: i64 = 0;

    // Bonus pour le centre (colonne 3)
    for r in 0..6 {
        if game.grid[r][3] == player {
            score += 3;
        }
    }

    // Horizontales
    for r in 0..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r][c + 1], game.grid[r][c + 2], game.grid[r][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Verticales
    for r in 0..3 {
        for c in 0..7 {
            let w = [game.grid[r][c], game.grid[r + 1][c], game.grid[r + 2][c], game.grid[r + 3][c]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-droite
    for r in 0..3 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r + 1][c + 1], game.grid[r + 2][c + 2], game.grid[r + 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    // Diagonales bas-gauche
    for r in 3..6 {
        for c in 0..4 {
            let w = [game.grid[r][c], game.grid[r - 1][c + 1], game.grid[r - 2][c + 2], game.grid[r - 3][c + 3]];
            score += evaluate_window(&w, player);
        }
    }

    score as f64
}
// Statistiques de recherche et Minimax (sans tri des coups).
#[allow(dead_code)]
#[derive(Default)]
struct Stats {
    nodes: u64,
}

// Minimax : les coups sont parcourus dans l'ordre brut des colonnes,
// et le premier maximum strict est retenu.
#[allow(dead_code)]
fn minimax(
    game: &mut Board,
    depth: i32,
    maximizing: bool,
    stats: &mut Stats,
    player: i32,
) -> (f64, Option<i32>) {
    stats.nodes += 1;

    if depth == 0 || game.is_terminal() {
        if game.is_terminal() {
            return (game.get_utility(player) * 1000.0, None);
        }
        return (evaluate_position(game, player), None);
    }

    let valid_moves = game.valid_moves();

    if maximizing {
        let mut best_value = f64::NEG_INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, false, stats, player);
            game.undo_move(mv);
            if value > best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    } else {
        let mut best_value = f64::INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = minimax(game, depth - 1, true, stats, player);
            game.undo_move(mv);
            if value < best_value {
                best_value = value;
                best_move = Some(mv);
            }
        }
        (best_value, best_move)
    }
}
// Alpha-Beta : les coups sont ordonnes par proximite au centre (tri stable),
// et la coupe s'applique quand beta <= alpha.
#[allow(dead_code)]
fn alpha_beta(
    game: &mut Board,
    depth: i32,
    mut alpha: f64,
    mut beta: f64,
    maximizing: bool,
    stats: &mut Stats,
    player: i32,
) -> (f64, Option<i32>) {
    stats.nodes += 1;

    if depth == 0 || game.is_terminal() {
        if game.is_terminal() {
            return (game.get_utility(player) * 1000.0, None);
        }
        return (evaluate_position(game, player), None);
    }

    let mut valid_moves = game.valid_moves();
    valid_moves.sort_by_key(|&c| (c - 3).abs());

    if maximizing {
        let mut best_value = f64::NEG_INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = alpha_beta(game, depth - 1, alpha, beta, false, stats, player);
            game.undo_move(mv);
            if value > best_value {
                best_value = value;
                best_move = Some(mv);
            }
            if best_value > alpha {
                alpha = best_value;
            }
            if beta <= alpha {
                break;
            }
        }
        (best_value, best_move)
    } else {
        let mut best_value = f64::INFINITY;
        let mut best_move = valid_moves.first().copied();
        for &mv in valid_moves.iter() {
            game.drop_piece(mv);
            let (value, _) = alpha_beta(game, depth - 1, alpha, beta, true, stats, player);
            game.undo_move(mv);
            if value < best_value {
                best_value = value;
                best_move = Some(mv);
            }
            if best_value < beta {
                beta = best_value;
            }
            if beta <= alpha {
                break;
            }
        }
        (best_value, best_move)
    }
}
// MT19937 compatible avec random.Random de CPython (init_by_array).
// Le carnet sème 42 avant chaque demonstration stochastique : c'est ce
// generateur qui rend les sorties reproductibles.
#[allow(dead_code)]
struct Mt19937 {
    mt: [u32; 624],
    idx: usize,
}

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32
                .wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30))
                .wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 {
        Mt19937::new_by_array(&[seed])
    }
    fn new_by_array(key: &[u32]) -> Mt19937 {
        let mut mt = Mt19937::init_genrand(19650218);
        let mut i: usize = 1;
        let mut j: usize = 0;
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j])
                .wrapping_add(j as u32);
            i += 1;
            j += 1;
            if i >= 624 {
                mt[0] = mt[623];
                i = 1;
            }
            if j >= key.len() {
                j = 0;
            }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 {
                mt[0] = mt[623];
                i = 1;
            }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 {
                    v ^= 0x9908_b0df;
                }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn getrandbits(&mut self, k: u32) -> u64 {
        if k <= 32 {
            return (self.next_u32() >> (32 - k)) as u64;
        }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    // Rejet, comme _randbelow de CPython.
    fn randbelow(&mut self, n: u64) -> u64 {
        if n == 0 {
            return 0;
        }
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n {
            r = self.getrandbits(k);
        }
        r
    }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T {
        &seq[self.randbelow(seq.len() as u64) as usize]
    }
}
// Monte Carlo Tree Search. L'arbre est un arena : chaque noeud porte son
// parent, ses enfants (dans l'ordre d'insertion) et ses compteurs.
#[allow(dead_code)]
#[derive(Clone)]
struct MctsNode {
    state: Board,
    action: Option<i32>,
    parent: Option<usize>,
    children: Vec<usize>,
    visits: u64,
    wins: f64,
    untried: Option<Vec<i32>>,
}

#[allow(dead_code)]
struct Mcts {
    c: f64,
    stats: Stats,
}

// Score UCB1 : infini tant que le noeud n'a pas ete visite.
#[allow(dead_code)]
fn ucb1(arena: &[MctsNode], index: usize, c: f64) -> f64 {
    let node = &arena[index];
    if node.visits == 0 {
        return f64::INFINITY;
    }
    let exploitation = node.wins / node.visits as f64;
    let parent_visits = match node.parent {
        Some(p) => arena[p].visits,
        None => 0,
    };
    if parent_visits == 0 {
        return f64::INFINITY;
    }
    let exploration = c * ((parent_visits as f64).ln() / node.visits as f64).sqrt();
    exploitation + exploration
}

// Premier maximum strict (comme max() de Python sur un dict ordonne).
#[allow(dead_code)]
fn best_child_ucb1(arena: &[MctsNode], index: usize, c: f64) -> usize {
    let mut best = arena[index].children[0];
    let mut best_score = ucb1(arena, best, c);
    for &child in arena[index].children.iter().skip(1) {
        let score = ucb1(arena, child, c);
        if score > best_score {
            best_score = score;
            best = child;
        }
    }
    best
}

// Premier maximum strict du nombre de visites (coup final).
#[allow(dead_code)]
fn best_child_visits(arena: &[MctsNode], index: usize) -> usize {
    let mut best = arena[index].children[0];
    let mut best_visits = arena[best].visits;
    for &child in arena[index].children.iter().skip(1) {
        if arena[child].visits > best_visits {
            best_visits = arena[child].visits;
            best = child;
        }
    }
    best
}

// Selection : descente UCB1 jusqu'a un noeud terminal ou non pleinement expandu.
#[allow(dead_code)]
fn select(
    arena: &mut Vec<MctsNode>,
    mut index: usize,
    game: &mut Board,
    c: f64,
    stats: &mut Stats,
) -> usize {
    loop {
        let state = arena[index].state.clone();
        game.set_state(state);
        if game.is_terminal() {
            return index;
        }
        if arena[index].untried.is_none() {
            let state = arena[index].state.clone();
            game.set_state(state);
            arena[index].untried = Some(game.valid_moves());
        }
        let untried_empty = arena[index].untried.as_ref().map(|v| v.is_empty()).unwrap_or(false);
        let has_children = !arena[index].children.is_empty();
        if !(untried_empty && has_children) {
            return expand(arena, index, game, stats);
        }
        let best = best_child_ucb1(arena, index, c);
        index = best;
        let state = arena[index].state.clone();
        game.set_state(state);
    }
}

// Expansion : le dernier coup restant est pris (LIFO), sans tirage.
#[allow(dead_code)]
fn expand(
    arena: &mut Vec<MctsNode>,
    index: usize,
    game: &mut Board,
    stats: &mut Stats,
) -> usize {
    stats.nodes += 1;
    if arena[index].untried.is_none() {
        let state = arena[index].state.clone();
        game.set_state(state);
        arena[index].untried = Some(game.valid_moves());
    }
    let action = match arena[index].untried.as_mut() {
        Some(list) => list.pop(),
        None => None,
    };
    let action = match action {
        Some(a) => a,
        None => return index,
    };
    let state = arena[index].state.clone();
    game.set_state(state);
    game.drop_piece(action);
    let child_state = game.state();
    arena.push(MctsNode {
        state: child_state,
        action: Some(action),
        parent: Some(index),
        children: Vec::new(),
        visits: 0,
        wins: 0.0,
        untried: None,
    });
    let child = arena.len() - 1;
    arena[index].children.push(child);
    child
}

// Simulation : partie aleatoire jusqu'a un etat terminal.
#[allow(dead_code)]
fn simulate(game: &mut Board, rng: &mut Mt19937, stats: &mut Stats) -> f64 {
    stats.nodes += 1;
    let saved = game.state();
    let original_player = game.current_player;
    while !game.is_terminal() {
        let moves = game.valid_moves();
        if moves.is_empty() {
            break;
        }
        let mv = *rng.choice(&moves);
        game.drop_piece(mv);
    }
    let result = game.get_utility(original_player);
    game.set_state(saved);
    result
}

// Backpropagation : le resultat est inverse selon le joueur du noeud.
#[allow(dead_code)]
fn backpropagate(arena: &mut Vec<MctsNode>, mut index: usize, result: f64) {
    loop {
        arena[index].visits += 1;
        let player = arena[index].state.current_player;
        if player == -1 {
            arena[index].wins += result;
        } else {
            arena[index].wins -= result;
        }
        match arena[index].parent {
            Some(parent) => index = parent,
            None => break,
        }
    }
}

#[allow(dead_code)]
impl Mcts {
    // L'etat du jeu du caller n'est jamais modifie : il est sauvegarde en
    // entree et restaure en sortie.
    fn search(
        &mut self,
        game: &mut Board,
        iterations: u32,
        rng: &mut Mt19937,
    ) -> (Option<i32>, f64) {
        let c = self.c;
        let mut stats = Stats { nodes: 0 };
        let initial = game.state();
        let mut arena: Vec<MctsNode> = Vec::new();
        arena.push(MctsNode {
            state: initial.clone(),
            action: None,
            parent: None,
            children: Vec::new(),
            visits: 0,
            wins: 0.0,
            untried: None,
        });

        for _ in 0..iterations {
            let node = select(&mut arena, 0, game, c, &mut stats);
            let result = simulate(game, rng, &mut stats);
            backpropagate(&mut arena, node, result);
        }

        game.set_state(initial);
        self.stats = stats;

        if arena[0].children.is_empty() {
            return (None, 0.0);
        }
        let best = best_child_visits(&arena, 0);
        let visits = arena[best].visits;
        let win_rate = if visits > 0 {
            arena[best].wins / visits as f64
        } else {
            0.0
        };
        (arena[best].action, win_rate)
    }
}
// Separateur de milliers, comme le format `:,` de Python.
#[allow(dead_code)]
fn thousands(n: u64) -> String {
    let s = n.to_string();
    let len = s.len();
    let mut out = String::new();
    for (i, ch) in s.chars().enumerate() {
        if i > 0 && (len - i) % 3 == 0 {
            out.push(',');
        }
        out.push(ch);
    }
    out
}

// Liste d'entiers au format de la repr() d'une liste Python.
#[allow(dead_code)]
fn py_list_i32(values: &[i32]) -> String {
    let items: Vec<String> = values.iter().map(|v| v.to_string()).collect();
    format!("[{}]", items.join(", "))
}

// Un Option imprime comme un Optional Python.
#[allow(dead_code)]
fn py_opt_i32(value: Option<i32>) -> String {
    match value {
        Some(v) => v.to_string(),
        None => "None".to_string(),
    }
}
#[allow(dead_code)]
#[derive(Clone, Copy)]
enum Agent {
    Random,
    Minimax4,
    AlphaBeta4,
    Mcts1000,
}

#[allow(dead_code)]
fn agent_move(agent: Agent, game: &mut Board, rng: &mut Mt19937) -> Option<i32> {
    match agent {
        Agent::Random => {
            let moves = game.valid_moves();
            if moves.is_empty() {
                None
            } else {
                Some(*rng.choice(&moves))
            }
        }
        Agent::Minimax4 => {
            let maximizing = game.current_player == 1;
            let mut stats = Stats { nodes: 0 };
            let (_, mv) = minimax(game, 4, maximizing, &mut stats, 1);
            mv
        }
        Agent::AlphaBeta4 => {
            let maximizing = game.current_player == 1;
            let mut stats = Stats { nodes: 0 };
            let (_, mv) = alpha_beta(
                game,
                4,
                f64::NEG_INFINITY,
                f64::INFINITY,
                maximizing,
                &mut stats,
                1,
            );
            mv
        }
        Agent::Mcts1000 => {
            let mut mcts = Mcts { c: 1.41, stats: Stats { nodes: 0 } };
            let (mv, _) = mcts.search(game, 1000, rng);
            mv
        }
    }
}

// 1 : agent1 gagne, -1 : agent2 gagne, 0 : nul. Un coup invalide est une
// defaite de l'agent qui joue.
#[allow(dead_code)]
fn play_game(agent1: Agent, agent2: Agent, rng: &mut Mt19937) -> i32 {
    let mut game = Board::new();
    while !game.is_terminal() {
        let mv = if game.current_player == 1 {
            agent_move(agent1, &mut game, rng)
        } else {
            agent_move(agent2, &mut game, rng)
        };
        match mv {
            Some(m) if game.valid_moves().contains(&m) => {
                game.drop_piece(m);
            }
            _ => return -game.current_player,
        }
    }
    game.check_winner()
}

let names = ["Random", "Minimax(d=4)", "AlphaBeta(d=4)", "MCTS(1000)"];
let agents = [
    Agent::Random,
    Agent::Minimax4,
    Agent::AlphaBeta4,
    Agent::Mcts1000,
];
let n_games = 6usize;

// Une seule graine pour tout le tournoi : le flux de tirage est partage par
// l'agent aleatoire et par les rollouts MCTS.
let mut rng = Mt19937::new(42);
let mut wins = [0u32; 4];
let mut losses = [0u32; 4];
let mut draws = [0u32; 4];

println!("Lancement du tournoi (6 parties par match)...");
for i in 0..4usize {
    for j in (i + 1)..4usize {
        let (mut w1, mut w2, mut d) = (0u32, 0u32, 0u32);
        for g in 0..n_games {
            // Qui commence alterne d'une partie a l'autre ; les scores restent
            // dans le sens name1 / name2.
            if g < n_games / 2 {
                let result = play_game(agents[i], agents[j], &mut rng);
                if result == 1 {
                    w1 += 1;
                } else if result == -1 {
                    w2 += 1;
                } else {
                    d += 1;
                }
            } else {
                let result = play_game(agents[j], agents[i], &mut rng);
                if result == 1 {
                    w2 += 1;
                } else if result == -1 {
                    w1 += 1;
                } else {
                    d += 1;
                }
            }
        }
        wins[i] += w1;
        losses[i] += w2;
        draws[i] += d;
        wins[j] += w2;
        losses[j] += w1;
        draws[j] += d;
        println!(
            "Match: {} vs {}... {}-{}-{}",
            names[i], names[j], w1, w2, d
        );
    }
}

// Tri par taux de victoire decroissant ; le tri est stable, donc l'ordre
// d'insertion tranche les egalites.
let mut order: Vec<usize> = (0..4).collect();
order.sort_by(|&a, &b| {
    let total_a = wins[a] + losses[a] + draws[a];
    let total_b = wins[b] + losses[b] + draws[b];
    let wr_a = wins[a] as f64 / total_a as f64;
    let wr_b = wins[b] as f64 / total_b as f64;
    wr_b.partial_cmp(&wr_a).unwrap_or(std::cmp::Ordering::Equal)
});

println!();
println!("Resultats du tournoi:");
{
    for &i in order.iter() {
        let total = wins[i] + losses[i] + draws[i];
        let wr = wins[i] as f64 / total as f64 * 100.0;
        println!(
            "{:<17}{:>2}V {:>2}D {:>2}N  WR {:.6}%",
            names[i], wins[i], losses[i], draws[i], wr
        );
    }
};


Lancement du tournoi (6 parties par match)...


Match: Random vs Minimax(d=4)... 0-6-0


Match: Random vs AlphaBeta(d=4)... 0-6-0


Match: Random vs MCTS(1000)... 4-2-0


Match: Minimax(d=4) vs AlphaBeta(d=4)... 0-6-0


Match: Minimax(d=4) vs MCTS(1000)... 6-0-0


... (2 outputs left out)

AlphaBeta(d=4)   18V  0D  0N  WR 100.000000%


Minimax(d=4)     12V  6D  0N  WR 66.666667%


Random            4V 14D  0N  WR 22.222222%


MCTS(1000)        2V 16D  0N  WR 11.111111%


### Interprétation : Résultats du tournoi

**Sortie obtenue** : Tournoi round-robin entre 4 agents (6 parties par match).

| Agent | Victoires | Defaites | Win Rate | Performance |
|-------|-----------|----------|----------|-------------|
| AlphaBeta(d=4) | 18 | 0 | 100.0% | **Dominant** |
| Minimax(d=4) | 12 | 6 | 66.7% | Solide |
| Random | 4 | 14 | 22.2% | Faible |
| MCTS(1000) | 2 | 16 | 11.1% | Très faible |

**Points clés** :
1. **Alpha-Beta domine sans perdre une partie** : 18-0, balayage 6-0 contre les trois autres agents
2. **Minimax solide** : 66.7% de win rate, ne perd que contre Alpha-Beta (0-6), balaie Random et MCTS (6-0 chacun)
3. **MCTS en difficulté** : dernier du tournoi, perd contre Alpha-Beta (0-6), Minimax (0-6) et même contre Random (2-4) -- 1000 itérations restent peu pour Connect Four
4. **échantillon insuffisant** : 6 parties par match = résultats variables

Points forts / faibles par algorithme :

| Algorithme | Points forts | Points faibles | Usage recommande |
|------------|--------------|----------------|------------------|
| **Minimax** | Simple, optimal | Explosion combinatoire | Petits jeux, enseignement |
| **Alpha-Beta** | efficace, optimal | Ordonnancement critique | Jeux a branchement moyen |
| **MCTS** | Pas d'heuristique, parallèle | probabiliste | grands espaces, jeux complexes |

**observations clés** :

1. **Alpha-Beta** réduit drastiquement le nombre de noeuds (facteur 5-20x selon la profondeur)
2. **MCTS** converge vers de bons coups sans fonction d'évaluation explicite

> **Note technique** : Le tournoi est **semé** (`Mt19937::new(42)` au début de la cellule du tournoi, #9434) : ces scores sont reproductibles a la re-exécution, comme ceux du jumeau Python. La différence entre Minimax et Alpha-Beta devrait etre nulle en théorie (même profondeur = même coups) ; le 6-0 d'Alpha-Beta sur Minimax suggere une différence d'ordonnancement des coups dans l'elagage. Avec 50+ parties (exercice 5), les win rates des duels équilibrés convergeraient vers ~50% pour Minimax vs Alpha-Beta.

## Exercices

Les six exercices ci-dessous vous font manipuler les trois familles d'agents (Minimax, Alpha-Beta, MCTS) vues plus haut. Ils progressent du reglage d'un hyperparamètre (1, 2) a l'amélioration de l'évaluation (3), puis aux techniques de recherche avancées (4, 6) et a la validation expérimentale (5). Chaque stub est autonome : sous `evcxr` chaque cellule est compilée isolement et redéfinit ce qu'elle utilise -- le moteur `Board`, l'heuristique `evaluate_position` et les recherches `minimax` et `alpha_beta` sont ainsi réécrits dans la cellule qui les appelle.

### Exercice 1 -- Profondeur variable vs temps

**objectif** : comparer Alpha-Beta aux profondeurs 4, 6 et 8, et déterminer la profondeur optimale pour un budget de 1 seconde. C'est le levier classique du compromis temps/profondeur en recherche adverse.

**Indices** : `run_benchmark_depth()` (section 6) comme base ; mesurer temps + nombre de noeuds ; faire jouer les algorithmes entre eux pour juger la qualité des coups, pas seulement la vitesse.

In [13]:
// TODO etudiant : comparer Alpha-Beta aux profondeurs 4, 6 et 8, mesurer le
// temps et le nombre de noeuds pour chaque profondeur, analyser la qualite
// des coups, puis determiner la profondeur optimale pour un budget de 1
// seconde.
// Indice : utiliser le benchmark par profondeur comme base.
#[allow(dead_code)]
fn benchmark_profondeur_optimale(_temps_cible: f64) -> Option<Vec<(usize, f64)>> {
    // TODO: Implementer la recherche de profondeur optimale
    None
}

println!("Exercice a completer - voir les indices ci-dessus");


Exercice a completer - voir les indices ci-dessus


### Exercice 2 -- Paramètre d'exploration c de MCTS

**objectif** : tester l'impact du coefficient c de UCB1 (`c = 0.5` exploitation, `1.0` équilibre, `1.41 = sqrt(2)` théorie, `2.0` exploration) sur la force de jeu de MCTS.

**Indices** : 20 parties contre un agent random pour chaque valeur de c ; le win rate trace la courbe exploration/exploitation.

In [14]:
// TODO etudiant : tester differentes valeurs de c (0.5 exploitation
// dominante, 1.0 equilibre, 1.41 valeur theorique, 2.0 exploration
// dominante) et analyser l'impact sur la force de jeu de MCTS.
// Indice : jouer 20 parties contre un agent random pour chaque valeur de c.
#[allow(dead_code)]
fn benchmark_mcts_c(_valeurs_c: &[f64], _parties: usize) -> Vec<(f64, f64)> {
    // TODO: Implementer le benchmark
    Vec::new()
}

println!("Exercice a completer - voir les indices ci-dessus");


Exercice a completer - voir les indices ci-dessus


### Exercice 3 -- Heuristique améliorée (détection de menaces)

**objectif** : enrichir la fonction d'évaluation avec la détection des menaces imminentes (3 jetons alignes + 1 case vide accessible au-dessus) et comparer a l'heuristique de base.

**Indices** : une fenêtre gagnante = 3 jetons + 1 case vide au sommet d'une colonne ; bonus/malus important pour ces menaces car elles décident le coup suivant.

In [15]:
// TODO etudiant : ajouter la detection des menaces imminentes (3 jetons
// alignes et 1 case vide accessible au-dessus) a l'evaluation, puis comparer
// avec l'heuristique de base.
// Indice : une fenetre gagnante est 3 jetons + 1 case vide au-dessus.
#[allow(dead_code)]
fn evaluate_position_improved(_plateau: &[[i32; 7]; 6], _player: i32) -> f64 {
    // TODO: Implementer l'evaluation amelioree et la detection des menaces
    0.0
}

println!("Exercice a completer - voir les indices ci-dessus");


Exercice a completer - voir les indices ci-dessus


### Exercice 4 -- Itérative Deepening avec limite de temps

**objectif** : implementer une recherche Alpha-Beta qui démarre a profondeur 1 et augmente progressivement jusqu'a épuisément du budget temps, en retournant le meilleur coup de la dernière profondeur complète.

**Indices** : `time.time()` pour mesurer le temps ecoule ; on garde le coup de la profondeur N-1 si la profondeur N est interrompue. C'est la technique utilisée en pratique pour respecter un budget temps strict.

In [16]:
// TODO etudiant : implementer une recherche Alpha-Beta qui demarre a
// profondeur 1 et augmente progressivement jusqu'a epuisement du budget
// temps, en retournant le meilleur coup de la derniere profondeur complete.
// Indice : mesurer le temps ecoule ; on garde le coup de la profondeur N-1 si
// la profondeur N est interrompue.
#[allow(dead_code)]
fn alphabeta_iterative_deepening(
    _plateau: &[[i32; 7]; 6],
    _temps_max: f64,
) -> Option<(i32, i32, f64)> {
    // TODO: Implementer l'iterative deepening
    None
}

println!("Exercice a completer - voir les indices ci-dessus");


Exercice a completer - voir les indices ci-dessus


### Exercice 5 -- Tournoi etendu avec analyse statistique

**objectif** : lancer un tournoi robuste (50 parties par match) entre Random, Minimax(d=4), AlphaBeta(d=4) et MCTS(1000), avec intervalles de confiance et test de significativite sur les win rates.

**Indices** : `scipy.stats` pour les tests statistiques ; 50 parties reduisent la variance pour distinguer les agents dont les win rates sont proches.

In [17]:
// TODO etudiant : lancer un tournoi robuste (50 parties par match) entre
// Random, Minimax(d=4), AlphaBeta(d=4) et MCTS(1000), avec intervalles de
// confiance et test de significativite sur les win rates.
// Indice : 50 parties reduisent la variance pour distinguer les agents dont
// les win rates sont proches.
#[allow(dead_code)]
fn tournoi_robuste(
    _agents: &[&str],
    _parties_par_match: usize,
) -> Vec<(String, u32, u32, u32)> {
    // TODO: Implementer le tournoi avec analyse statistique
    Vec::new()
}

println!("Exercice a completer - voir les indices ci-dessus");


Exercice a completer - voir les indices ci-dessus


### Exercice 6 -- Livre d'ouvertures (Opening Book)

**objectif** : precalculer les meilleurs premiers coups avec un Alpha-Beta profond (d=8), les stocker dans un dictionnaire `{plateau_hash: meilleur_coup}`, puis mesurer le gain (temps economise + qualité) en debut de partie.

**Indices** : un `dict {(plateau_hash): meilleur_coup}` ; la génération se fait une fois pour toutes (hors-ligne), la consultation en match est O(1).

In [18]:
// TODO etudiant : precalculer les meilleurs premiers coups avec un Alpha-Beta
// profond (d=8), les stocker dans une table {plateau_hash: meilleur_coup},
// puis mesurer le gain (temps economise et qualite) en debut de partie.
// Indice : la generation se fait une fois pour toutes (hors-ligne), la
// consultation en match est immediate.
#[allow(dead_code)]
struct OpeningBook {
    coups: std::collections::HashMap<String, i32>,
}

#[allow(dead_code)]
impl OpeningBook {
    fn new() -> OpeningBook {
        OpeningBook {
            coups: std::collections::HashMap::new(),
        }
    }
    fn lookup(&self, _plateau: &[[i32; 7]; 6]) -> Option<i32> {
        // TODO: Implementer la recherche
        None
    }
    fn generate(&mut self, _max_moves: usize) {
        // TODO: Implementer la generation
    }
}

println!("Exercice a completer - voir les indices ci-dessus");


Exercice a completer - voir les indices ci-dessus


## Conclusion

### Synthese comparative

| critère | Minimax | Alpha-Beta | MCTS |
|---------|---------|------------|------|
| **Optimalité** | Oui (si profondeur complète) | Oui (identique a Minimax) | probabiliste |
| **Complexité** | O(b^d) | O(b^(d/2)) optimal | O(n * d) |
| **Fonction d'évaluation** | nécessaire | nécessaire | Non requise |
| **Parallelisation** | Difficile | Difficile | très facile |
| **Meilleur usage** | Petits jeux | Jeux moyens | grands jeux |

### Recommandations pratiques

1. **Pour Connect Four** : Alpha-Beta avec profondeur 6-8 et bonne heuristique
2. **Pour les échecs** : Alpha-Beta avec tables de transposition
3. **Pour le Go** : MCTS + réseaux de neurones (AlphaGo)
4. **Pour un temps limite strict** : MCTS ou itérative deepening

### Pour aller plus loin

- **Search-6** : Approfondir Alpha-Beta, transposition tables
- **Search-7** : MCTS avance, OpenSpiel, AlphaZero
- **App-12** : 8 algorithmes compares sur Connect Four
- **GameTheory** : Jeux a information imparfaite, Nash equilibrium

***

**Navigation** : [<< App-12 ConnectFour](App-12-ConnectFour-Python.ipynb) | [Index](../README.md) | [CSP Applications >>](App-01-NQueens-Python.ipynb) | [Version Python](App-14-ConnectFour-Adversarial-Python.ipynb) | [Version C#](App-14-ConnectFour-Adversarial-CSharp.ipynb)